In [1]:
# =====================================
# CELL 1: SETUP & INSTALLATION
# =====================================

print("🚀 GPU-Enhanced Scribble Plotter - Modular Cell Architecture")
print("🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research")
print("="*60)

# Install packages
!pip install -q matplotlib numpy pandas regex ipywidgets ezdxf reportlab tqdm scikit-learn torch torchvision

print("✅ Packages installed!")

# =====================================
# CELL 2: IMPORTS & GLOBALS
# =====================================

import os, sys, json, re, random, math, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive, files
import torch

# Global variables for inter-cell communication
SYSTEM_CONFIG = {}
COMPONENTS = {}

print("✅ Imports complete!")

# =====================================
# CELL 3: SYSTEM SETUP FUNCTIONS
# =====================================

def setup_gpu_system():
    """Setup GPU and directory structure"""
    global SYSTEM_CONFIG

    # GPU setup
    gpu_available = torch.cuda.is_available()
    device = torch.device('cuda' if gpu_available else 'cpu')

    print(f"🚀 GPU: {'✅ Available' if gpu_available else '❌ Not Available'}")
    if gpu_available:
        print(f"⚡ Device: {torch.cuda.get_device_name(0)}")

    # Mount drive
    try:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
    except:
        print("⚠️ Drive already mounted")

    # Create directories with GROUP support
    base_path = '/content/drive/MyDrive/ScribblePlotter_Organized'
    directories = {
        'base': base_path,
        'input': f'{base_path}/input',
        'output': f'{base_path}/output',
        'config': f'{base_path}/config',
        'group_pdf': f'{base_path}/output/GROUP_PDF',
        'group_dxf': f'{base_path}/output/GROUP_DXF',
        'group_png': f'{base_path}/output/GROUP_PNG'
    }

    for name, path in directories.items():
        os.makedirs(path, exist_ok=True)
        print(f"📁 {name}")

    SYSTEM_CONFIG = {
        'gpu_available': gpu_available,
        'device': device,
        'directories': directories,
        'total_examples': 3,
        'generate_pdf': True,
        'generate_dxf': True,
        'generate_png': True,
        'organize_groups': True,
        'use_ai': True,
        'use_hopfield': True
    }

    print("✅ System setup complete!")
    return SYSTEM_CONFIG

# Run setup
setup_gpu_system()

# =====================================
# CELL 4: PLT PROCESSOR CLASS
# =====================================

class PLTProcessor:
    """Handles PLT file processing with GROUP directory awareness"""

    def __init__(self):
        self.patterns = {
            'PD': re.compile(r'PD(\d+),(\d+)'),
            'PD_neg_x': re.compile(r'PD-(\d+),(\d+)'),
            'PD_neg_y': re.compile(r'PD(\d+),-(\d+)'),
            'PD_neg_both': re.compile(r'PD-(\d+),-(\d+)'),
            'PA': re.compile(r'PA(\d+)\.(\d+),(\d+)\.(\d+)'),
            'PU': re.compile(r'PU(\d+)\.(\d+),(\d+)\.(\d+)')
        }

    def process_file(self, file_path):
        """Process PLT file and return coordinates"""
        try:
            with open(file_path, 'r') as f:
                content = f.read()

            if 'PW0' in content[:200]:
                lines = content.replace('\n', '').replace('\r', '').split(';')
            else:
                lines = content.split(';')

            coordinates = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                for pattern_name, pattern in self.patterns.items():
                    match = pattern.match(line)
                    if match:
                        coord = self._extract_coord(pattern_name, match)
                        if coord:
                            coordinates.append(coord)
                        break

            coordinates.reverse()
            return self._scale_coordinates(coordinates)

        except Exception as e:
            print(f"❌ PLT error: {e}")
            return []

    def _extract_coord(self, pattern_name, match):
        """Extract coordinate from regex match"""
        groups = match.groups()

        if pattern_name == 'PD':
            return (float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_x':
            return (-float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_y':
            return (float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_both':
            return (-float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PA':
            return (float(groups[0]), float(groups[2]), 0.0)
        elif pattern_name == 'PU':
            return (float(groups[0]), float(groups[2]), 10.0)
        return None

    def _scale_coordinates(self, coordinates):
        """Scale coordinates"""
        if not coordinates:
            return []

        scale_x, scale_y = 0.09, 0.09
        return [(x * scale_x, y * scale_y, z) for x, y, z in coordinates]

print("✅ PLTProcessor class defined")

# =====================================
# CELL 5: AI SYSTEM CLASS
# =====================================

class GPUAcceleratedAI:
    """AI system for parameter prediction"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']

        if self.gpu_available:
            torch.cuda.empty_cache()
        print(f"🧠 AI initialized ({'GPU' if self.gpu_available else 'CPU'})")

    def predict_parameters(self, points):
        """Predict scribble parameters from points"""
        try:
            features = self._extract_features(points)

            complexity = features[0]
            path_length = features[1]

            # AI parameter selection
            if complexity > 0.7:
                steps = random.randint(8, 15)
                scribble = random.uniform(1.0, 3.0)
            elif complexity > 0.4:
                steps = random.randint(4, 10)
                scribble = random.uniform(2.0, 5.0)
            else:
                steps = random.randint(2, 7)
                scribble = random.uniform(3.0, 7.0)

            # Adjust for path length
            if path_length > 2000:
                scribble *= 1.2
                steps = min(steps + 2, 15)
            elif path_length < 500:
                scribble *= 0.8
                steps = max(steps - 1, 2)

            return {
                'steps': max(2, min(20, steps)),
                'scribble': max(0.1, min(10.0, scribble)),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': complexity
            }

        except Exception as e:
            print(f"❌ AI error: {e}")
            return {
                'steps': random.randint(3, 8),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': 0.5
            }

    def _extract_features(self, points):
        """Extract features from points"""
        if not points:
            return np.zeros(15)

        coords = np.array([(p[0], p[1]) for p in points if p[2] == 0])

        if len(coords) < 2:
            return np.zeros(15)

        features = [0.5] * 15

        # Basic calculations
        diffs = np.diff(coords, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        features[1] = np.sum(distances)  # Total length

        # Bounding box
        if len(coords) > 0:
            min_coords = np.min(coords, axis=0)
            max_coords = np.max(coords, axis=0)
            features[2] = max_coords[0] - min_coords[0]  # Width
            features[3] = max_coords[1] - min_coords[1]  # Height
            features[4] = features[2] / max(features[3], 1e-6)  # Aspect ratio

        # Complexity (simplified)
        if len(distances) > 1:
            features[0] = np.std(distances) / np.mean(distances)

        return np.array(features)

print("✅ GPUAcceleratedAI class defined")

# =====================================
# CELL 6: HOPFIELD NETWORK CLASS
# =====================================

class HopfieldNetwork:
    """Hopfield network for style memory - Kent Benson's 1983-1986 vision realized"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']
        self.pattern_size = 15
        self.memory_size = 200

        if self.gpu_available:
            self.weights = torch.zeros(self.pattern_size, self.pattern_size, device=self.device)
        else:
            self.weights = np.zeros((self.pattern_size, self.pattern_size))

        self.stored_patterns = []
        self.pattern_labels = []
        print("🧠 Hopfield network initialized")

    def store_pattern(self, pattern, label=""):
        """Store a pattern in memory"""
        try:
            if len(self.stored_patterns) >= self.memory_size:
                return False

            if self.gpu_available:
                norm_pattern = self._normalize_gpu(pattern)
                self.stored_patterns.append(norm_pattern.cpu().numpy())
                self.weights += torch.outer(norm_pattern, norm_pattern)
            else:
                norm_pattern = self._normalize_cpu(pattern)
                self.stored_patterns.append(norm_pattern)
                self.weights += np.outer(norm_pattern, norm_pattern)

            self.pattern_labels.append(label)
            return True

        except Exception as e:
            print(f"❌ Store pattern error: {e}")
            return False

    def recall_pattern(self, partial_pattern, max_iterations=50):
        """Recall pattern from partial input"""
        try:
            if self.gpu_available:
                current = self._normalize_gpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.clone()
                    activations = torch.mv(self.weights, current)
                    current = torch.sign(activations)

                    if torch.equal(current, previous):
                        break

                return ((current + 1) / 2).cpu().numpy()
            else:
                current = self._normalize_cpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.copy()

                    for i in range(len(current)):
                        activation = np.dot(self.weights[i], current)
                        current[i] = 1 if activation > 0 else -1

                    if np.array_equal(current, previous):
                        break

                return (current + 1) / 2

        except Exception as e:
            print(f"❌ Recall error: {e}")
            return partial_pattern

    def _normalize_gpu(self, pattern):
        """Normalize pattern for GPU"""
        pattern_tensor = torch.tensor(pattern[:self.pattern_size],
                                    dtype=torch.float32, device=self.device)

        if len(pattern_tensor) < self.pattern_size:
            padding = torch.zeros(self.pattern_size - len(pattern_tensor), device=self.device)
            pattern_tensor = torch.cat([pattern_tensor, padding])

        if torch.std(pattern_tensor) > 0:
            pattern_tensor = (pattern_tensor - torch.mean(pattern_tensor)) / torch.std(pattern_tensor)

        return torch.sign(pattern_tensor)

    def _normalize_cpu(self, pattern):
        """Normalize pattern for CPU"""
        pattern = np.array(pattern[:self.pattern_size])
        if len(pattern) < self.pattern_size:
            pattern = np.pad(pattern, (0, self.pattern_size - len(pattern)))

        if np.std(pattern) > 0:
            pattern = (pattern - np.mean(pattern)) / np.std(pattern)

        return np.sign(pattern)

print("✅ HopfieldNetwork class defined")

# =====================================
# CELL 7: RENDERER CLASS
# =====================================

class ScribbleRenderer:
    """Renders scribble artwork"""

    def __init__(self):
        self.page_width = 800
        self.page_height = 600
        print("🎨 Renderer initialized")

    def render_artwork(self, points, params):
        """Render scribble artwork"""
        try:
            if not points:
                return None

            fig, ax = plt.subplots(figsize=(self.page_width/100, self.page_height/100))
            ax.set_aspect('equal')
            ax.set_facecolor('white')
            ax.axis('off')

            steps = params['steps']
            scrib_val = params['scribble']
            stroke_weight = params['stroke_weight']
            color = params['color']

            # Render with scribble algorithm
            for i in range(len(points) - 1):
                x1, y1, z1 = points[i]

                if z1 == 0:  # Pen down
                    x2, y2, z2 = points[i + 1] if i + 1 < len(points) else points[i]

                    x_step = (x2 - x1) / steps
                    y_step = (y2 - y1) / steps
                    current_x, current_y = x1, y1

                    for step in range(steps):
                        if step < steps - 1:
                            next_x = current_x + x_step + random.uniform(-scrib_val, scrib_val)
                            next_y = current_y + y_step + random.uniform(-scrib_val, scrib_val)
                        else:
                            next_x, next_y = x2, y2

                        ax.plot([current_x, next_x], [current_y, next_y],
                               color=color, linewidth=stroke_weight/5, alpha=0.8)

                        current_x, current_y = next_x, next_y

            ax.set_xlim(auto=True)
            ax.set_ylim(auto=True)
            ax.invert_yaxis()
            plt.tight_layout()

            return fig

        except Exception as e:
            print(f"❌ Render error: {e}")
            return None

print("✅ ScribbleRenderer class defined")

# =====================================
# CELL 8: DIRECTORY MANAGER CLASS
# =====================================

class DirectoryManager:
    """Manages file organization and GROUP directories"""

    def __init__(self):
        self.processed_files = []
        self.file_registry = {}
        print("📁 Directory manager initialized")

    def create_working_directories(self, base_dir, file_base_name, file_index):
        """Create working directories - Python version of createDirectories_scribble_plotter"""
        try:
            working_dir = Path(base_dir) / f"{file_base_name}_{file_index}"

            directories = {
                'pdf': working_dir / "PDF",
                'dxf': working_dir / "DXF",
                'png': working_dir / "PNG",
                'tmp': working_dir / "TMP",
                'group_tiff': Path(base_dir) / "GROUP_TIFF",
                'group_dxf': Path(base_dir) / "GROUP_DXF",
                'group_pdf': Path(base_dir) / "GROUP_PDF",
                'group_png': Path(base_dir) / "GROUP_PNG",
                'group_transform': Path(base_dir) / "GROUP_TRANSFORM"
            }

            # Create all directories
            for name, path in directories.items():
                path.mkdir(parents=True, exist_ok=True)

            return {name: str(path) for name, path in directories.items()}

        except Exception as e:
            print(f"❌ Directory creation error: {e}")
            return {}

    def organize_output_files(self):
        """Copy all generated files to group directories"""
        try:
            base_output = Path(SYSTEM_CONFIG['directories']['output'])

            for file_path in self.processed_files:
                source = Path(file_path)
                if not source.exists():
                    continue

                # Determine group directory based on file extension
                if source.suffix.lower() == '.pdf':
                    group_dir = base_output / "GROUP_PDF"
                elif source.suffix.lower() == '.dxf':
                    group_dir = base_output / "GROUP_DXF"
                elif source.suffix.lower() == '.png':
                    group_dir = base_output / "GROUP_PNG"
                else:
                    continue

                group_dir.mkdir(exist_ok=True)
                destination = group_dir / source.name

                # Copy file
                shutil.copy2(source, destination)

            print(f"📦 Organized {len(self.processed_files)} files into GROUP directories")

        except Exception as e:
            print(f"⚠️ Error organizing files: {e}")

    def register_file(self, file_path, file_type):
        """Register processed file for organization"""
        self.processed_files.append(file_path)
        self.file_registry[file_path] = {
            'type': file_type,
            'timestamp': datetime.now()
        }

print("✅ DirectoryManager class defined")

# =====================================
# CELL 9: INITIALIZE COMPONENTS
# =====================================

# Initialize all components
COMPONENTS = {
    'plt_processor': PLTProcessor(),
    'ai_system': GPUAcceleratedAI(),
    'hopfield_network': HopfieldNetwork(),
    'renderer': ScribbleRenderer(),
    'directory_manager': DirectoryManager()
}

print("✅ All components initialized and ready!")
print(f"🚀 GPU: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")
print(f"📁 Base directory: {SYSTEM_CONFIG['directories']['base']}")

# =====================================
# CELL 10: UTILITY FUNCTIONS
# =====================================

def get_plt_files(directory):
    """Get all PLT files in directory"""
    try:
        path = Path(directory)
        plt_files = list(path.glob("*.plt")) + list(path.glob("*.PLT"))
        print(f"📁 Found {len(plt_files)} PLT files")
        return plt_files
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def save_dxf_simple(points, output_path, params):
    """Simple DXF save with scribble algorithm"""
    try:
        import ezdxf
        doc = ezdxf.new('R2010')
        msp = doc.modelspace()

        steps = params['steps']
        scrib_val = params['scribble']

        for i in range(len(points) - 1):
            x1, y1, z1 = points[i]
            x2, y2, z2 = points[i + 1]

            if z1 == 0:  # Pen down
                x_step = (x2 - x1) / steps
                y_step = (y2 - y1) / steps
                prev_x, prev_y = x1, y1

                for step in range(steps):
                    if step < steps - 1:
                        curr_x = prev_x + x_step + random.uniform(-scrib_val, scrib_val)
                        curr_y = prev_y + y_step + random.uniform(-scrib_val, scrib_val)
                    else:
                        curr_x, curr_y = x2, y2

                    msp.add_line((prev_x, prev_y), (curr_x, curr_y))
                    prev_x, prev_y = curr_x, curr_y

        doc.saveas(output_path)
        return True
    except Exception as e:
        print(f"DXF error: {e}")
        return False

def update_config(key, value):
    """Update system configuration"""
    global SYSTEM_CONFIG
    SYSTEM_CONFIG[key] = value
    print(f"🔧 Updated {key} = {value}")

def gpu_status():
    """Check GPU status"""
    if SYSTEM_CONFIG['gpu_available']:
        print("🚀 GPU STATUS: AVAILABLE")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"   Memory: {allocated:.2f} GB / {total:.2f} GB")
    else:
        print("❌ GPU STATUS: NOT AVAILABLE")

print("✅ Utility functions defined")

# =====================================
# CELL 11: PROCESSING FUNCTIONS
# =====================================

def process_single_file(plt_file, iteration=0):
    """Process a single PLT file with complete AI enhancement"""
    try:
        print(f"\n🎨 Processing: {plt_file.name} (iteration {iteration + 1})")

        # 1. Process PLT file
        points = COMPONENTS['plt_processor'].process_file(str(plt_file))
        if not points:
            print("  ❌ Failed to process PLT")
            return False

        print(f"  📊 Loaded {len(points)} points")

        # 2. AI parameter prediction
        if SYSTEM_CONFIG['use_ai']:
            params = COMPONENTS['ai_system'].predict_parameters(points)
            print(f"  🧠 AI params: steps={params['steps']}, scribble={params['scribble']:.1f}")

            # 3. Hopfield enhancement
            if SYSTEM_CONFIG['use_hopfield']:
                features = COMPONENTS['ai_system']._extract_features(points)
                evolved = COMPONENTS['hopfield_network'].recall_pattern(features)

                # Blend AI with Hopfield evolution
                blend_factor = 0.7
                params['scribble'] = (blend_factor * params['scribble'] +
                                    (1 - blend_factor) * abs(evolved[0]) * 8)
                params['scribble'] = max(0.1, min(10.0, params['scribble']))
                print(f"  🧠 Hopfield enhanced: scribble={params['scribble']:.1f}")
        else:
            # Traditional random parameters
            params = {
                'steps': random.randint(3, 10),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random())
            }

        # 4. Create directories
        base_name = plt_file.stem
        output_base = SYSTEM_CONFIG['directories']['output']
        dirs = COMPONENTS['directory_manager'].create_working_directories(
            output_base, base_name, iteration)

        # 5. Render artwork
        figure = COMPONENTS['renderer'].render_artwork(points, params)
        if not figure:
            print("  ❌ Render failed")
            return False

        # 6. Save outputs
        method_suffix = params.get('generation_method', 'ai')
        output_name = f"{base_name}_{method_suffix}_steps{params['steps']}_scribble{int(params['scribble'])}_{iteration}"
        success_count = 0

        # Save PNG
        if SYSTEM_CONFIG['generate_png']:
            png_path = Path(dirs['png']) / f"{output_name}.png"
            try:
                figure.savefig(str(png_path), format='png', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(png_path), 'png')
                success_count += 1
                print("  ✅ PNG saved")
            except Exception as e:
                print(f"  ❌ PNG error: {e}")

        # Save PDF
        if SYSTEM_CONFIG['generate_pdf']:
            pdf_path = Path(dirs['pdf']) / f"{output_name}.pdf"
            try:
                figure.savefig(str(pdf_path), format='pdf', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(pdf_path), 'pdf')
                success_count += 1
                print("  ✅ PDF saved")
            except Exception as e:
                print(f"  ❌ PDF error: {e}")

        # Save DXF
        if SYSTEM_CONFIG['generate_dxf']:
            dxf_path = Path(dirs['dxf']) / f"{output_name}.dxf"
            try:
                if save_dxf_simple(points, str(dxf_path), params):
                    COMPONENTS['directory_manager'].register_file(str(dxf_path), 'dxf')
                    success_count += 1
                    print("  ✅ DXF saved")
            except Exception as e:
                print(f"  ❌ DXF error: {e}")

        plt.close(figure)

        # 7. Store successful style in Hopfield memory
        if success_count > 0 and SYSTEM_CONFIG['use_hopfield']:
            features = COMPONENTS['ai_system']._extract_features(points)
            style_name = f"{base_name}_{iteration}"
            COMPONENTS['hopfield_network'].store_pattern(features, style_name)

        if success_count > 0:
            print(f"  ✅ Generated {success_count} format(s) successfully")
            return True
        else:
            print("  ❌ Failed to save any formats")
            return False

    except Exception as e:
        print(f"  ❌ Processing error: {e}")
        return False

def process_batch(input_directory=None):
    """Process all PLT files in batch"""
    if input_directory is None:
        input_directory = SYSTEM_CONFIG['directories']['input']

    print("🚀 STARTING BATCH PROCESSING")
    print("="*50)

    # Get PLT files
    plt_files = get_plt_files(input_directory)
    if not plt_files:
        print("❌ No PLT files found")
        return {'success': False, 'message': 'No PLT files found'}

    total_examples = SYSTEM_CONFIG['total_examples']
    total_operations = len(plt_files) * total_examples

    print(f"📊 Processing {len(plt_files)} files × {total_examples} examples = {total_operations} operations")
    print(f"🚀 GPU Acceleration: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")

    successful_operations = 0

    # Process with progress bar
    with tqdm(total=total_operations, desc="Complete Processing") as pbar:
        for plt_file in plt_files:
            for iteration in range(total_examples):
                success = process_single_file(plt_file, iteration)
                if success:
                    successful_operations += 1

                pbar.update(1)
                pbar.set_description(f"Success: {successful_operations}/{total_operations}")

    # Organize files into GROUP directories
    if SYSTEM_CONFIG['organize_groups']:
        COMPONENTS['directory_manager'].organize_output_files()

    # Generate summary
    summary = {
        'success': True,
        'total_operations': total_operations,
        'successful_operations': successful_operations,
        'success_rate': f"{successful_operations}/{total_operations}",
        'hopfield_patterns': len(COMPONENTS['hopfield_network'].stored_patterns),
        'gpu_accelerated': SYSTEM_CONFIG['gpu_available'],
        'output_directory': SYSTEM_CONFIG['directories']['output']
    }

    print_batch_summary(summary)
    return summary

def print_batch_summary(summary):
    """Print batch processing summary"""
    print("\n" + "="*60)
    print("🎉 BATCH PROCESSING COMPLETED")
    print("="*60)
    print(f"⚡ Total operations: {summary['total_operations']}")
    print(f"✅ Successful: {summary['successful_operations']}")
    print(f"📊 Success rate: {summary['success_rate']}")
    print(f"🧠 Hopfield patterns learned: {summary['hopfield_patterns']}")
    print(f"🚀 GPU accelerated: {'YES' if summary['gpu_accelerated'] else 'NO'}")
    print(f"📁 Output directory: {summary['output_directory']}")
    print("="*60)

print("✅ Processing functions defined")

# =====================================
# CELL 12: USER INTERFACE & BINDING
# =====================================

def create_interface():
    """Create user interface that binds all components together"""

    # Interface functions
    def upload_files(button):
        """Upload PLT files"""
        with status_output:
            clear_output(wait=True)
            print("📤 Upload PLT Files")

            uploaded = files.upload()
            input_dir = Path(SYSTEM_CONFIG['directories']['input'])

            uploaded_count = 0
            for filename, content in uploaded.items():
                if filename.lower().endswith('.plt'):
                    with open(input_dir / filename, 'wb') as f:
                        f.write(content)
                    uploaded_count += 1
                    print(f"✅ Uploaded: {filename}")
                else:
                    print(f"⚠️ Skipped: {filename}")

            print(f"\n🎉 Uploaded {uploaded_count} PLT files!")

    def test_single_file(button):
        """Test with a single file"""
        with status_output:
            clear_output(wait=True)
            print("🧪 Testing single file...")

            plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
            if not plt_files:
                print("❌ No PLT files found")
                return

            print(f"🧪 Testing with: {plt_files[0].name}")
            success = process_single_file(plt_files[0], 999)

            if success:
                print("✅ Test completed successfully!")
                print(f"🧠 Hopfield patterns: {len(COMPONENTS['hopfield_network'].stored_patterns)}")
            else:
                print("❌ Test failed")

    def process_all_files(button):
        """Process all files"""
        with status_output:
            clear_output(wait=True)

            # Update config from widgets
            update_config('total_examples', examples_widget.value)
            update_config('use_ai', use_ai_widget.value)
            update_config('use_hopfield', use_hopfield_widget.value)
            update_config('generate_pdf', pdf_widget.value)
            update_config('generate_dxf', dxf_widget.value)
            update_config('generate_png', png_widget.value)
            update_config('organize_groups', organize_widget.value)

            result = process_batch()

            if result['success']:
                print("🎉 Batch processing completed successfully!")
            else:
                print("❌ Batch processing failed")

    def hopfield_demo(button):
        """Demonstrate Hopfield networks"""
        with status_output:
            clear_output(wait=True)

            print("🧠 HOPFIELD NETWORK DEMONSTRATION")
            print("Inspired by Kent Benson's 1983-1986 research")
            print("="*60)

            # Store demo patterns
            demo_patterns = [
                ([1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1], "geometric"),
                ([1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1], "alternating"),
                ([1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, 1, 1, 1], "grouped")
            ]

            for pattern, name in demo_patterns:
                COMPONENTS['hopfield_network'].store_pattern(pattern, name)

            print(f"\n🎨 Testing pattern recall...")
            # Test with noisy input
            noisy_input = [1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1]
            recalled = COMPONENTS['hopfield_network'].recall_pattern(noisy_input)

            print(f"🔍 Input (noisy):  {noisy_input}")
            print(f"🧠 Recalled:       {[f'{x:.1f}' for x in recalled]}")

            print("\n🎯 This demonstrates how neural networks can generate")
            print("   creative combinations beyond explicit training!")

    # Create widgets
    examples_widget = widgets.IntSlider(
        value=SYSTEM_CONFIG['total_examples'],
        min=1, max=10, step=1,
        description='Examples per file:',
        style={'description_width': 'initial'}
    )

    use_ai_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_ai'],
        description='🧠 Use AI Parameter Prediction'
    )

    use_hopfield_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_hopfield'],
        description='🧠 Use Hopfield Style Memory'
    )

    pdf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_pdf'],
        description='📄 Generate PDF'
    )

    dxf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_dxf'],
        description='📐 Generate DXF'
    )

    png_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_png'],
        description='🖼️ Generate PNG'
    )

    organize_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['organize_groups'],
        description='📦 Organize into GROUP directories'
    )

    # Control buttons
    upload_button = widgets.Button(
        description='📤 Upload PLT Files',
        button_style='primary',
        layout=widgets.Layout(width='200px', height='40px')
    )

    test_button = widgets.Button(
        description='🧪 Test Single File',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='40px')
    )

    process_button = widgets.Button(
        description='🚀 Process All Files',
        button_style='success',
        layout=widgets.Layout(width='200px', height='50px')
    )

    hopfield_demo_button = widgets.Button(
        description='🧠 Hopfield Demo',
        button_style='info',
        layout=widgets.Layout(width='200px', height='40px')
    )

    # Status output
    status_output = widgets.Output()

    # Bind events
    upload_button.on_click(upload_files)
    test_button.on_click(test_single_file)
    process_button.on_click(process_all_files)
    hopfield_demo_button.on_click(hopfield_demo)

    # Display interface
    gpu_status_text = "✅ ENABLED" if SYSTEM_CONFIG['gpu_available'] else "❌ DISABLED"
    hopfield_patterns = len(COMPONENTS['hopfield_network'].stored_patterns)

    header = widgets.HTML(f"""
    <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 20px;">
        <h1>🚀 Complete GPU-Integrated Scribble Plotter</h1>
        <p>Honoring Kent Benson's 1983-1986 Hopfield Network Research</p>
        <p><strong>GPU Acceleration: {gpu_status_text}</strong></p>
        <p><em>Neural networks controlling artistic generation in real-time</em></p>
    </div>
    """)

    status_info = widgets.HTML(f"""
    <div style="background: #f0f8ff; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3>📊 System Status</h3>
        <ul>
            <li><strong>GPU Available:</strong> {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}</li>
            <li><strong>Device:</strong> {SYSTEM_CONFIG['device']}</li>
            <li><strong>Hopfield Patterns:</strong> {hopfield_patterns} stored</li>
            <li><strong>Base Directory:</strong> {SYSTEM_CONFIG['directories']['base']}</li>
        </ul>
    </div>
    """)

    # Settings panels
    ai_settings = widgets.VBox([
        widgets.HTML("<h3>🧠 AI & Neural Network Settings</h3>"),
        use_ai_widget,
        use_hopfield_widget
    ])

    processing_settings = widgets.VBox([
        widgets.HTML("<h3>⚙️ Processing Settings</h3>"),
        examples_widget,
        organize_widget
    ])

    output_settings = widgets.VBox([
        widgets.HTML("<h3>📤 Output Settings</h3>"),
        pdf_widget,
        dxf_widget,
        png_widget
    ])

    controls = widgets.VBox([
        widgets.HTML("<h3>🎮 Controls</h3>"),
        widgets.HBox([upload_button, test_button]),
        widgets.HBox([process_button, hopfield_demo_button])
    ])

    # Main layout
    main_layout = widgets.VBox([
        header,
        status_info,
        widgets.HBox([
            widgets.VBox([ai_settings, processing_settings], layout=widgets.Layout(width='50%')),
            widgets.VBox([output_settings, controls], layout=widgets.Layout(width='50%'))
        ]),
        status_output
    ])

    display(main_layout)

# =====================================
# CELL 13: QUICK FUNCTIONS & TESTING
# =====================================

def quick_test():
    """Quick system test"""
    print("🧪 QUICK SYSTEM TEST")
    print("="*30)

    print(f"1. Configuration: {'✅ OK' if SYSTEM_CONFIG else '❌ FAILED'}")
    print(f"2. GPU Available: {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}")
    print(f"3. Components: {'✅ OK' if COMPONENTS else '❌ FAILED'}")

    # Test directories
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])
    print(f"4. Input Directory: {'✅ EXISTS' if input_dir.exists() else '❌ MISSING'}")

    if input_dir.exists():
        plt_files = get_plt_files(str(input_dir))
        print(f"5. PLT Files: {'✅ ' + str(len(plt_files)) if plt_files else '⚠️ NONE'}")

    print("\n💡 Use create_interface() to access full controls")
    print("💡 Use process_batch() to process files directly")

def hopfield_quick_demo():
    """Quick Hopfield demonstration"""
    print("🧠 HOPFIELD NETWORK QUICK DEMO")
    print("Kent Benson's 1986 vision of neural network art control")
    print("="*50)

    # Store a simple pattern
    artistic_pattern = [0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4, 0.5, 0.5, 0.4, 0.6, 0.3, 0.7, 0.2]
    COMPONENTS['hopfield_network'].store_pattern(artistic_pattern, "demo_style")

    # Test recall
    partial = [0.8, 0.2, 0.0, 0.0, 0.7, 0.0, 0.6, 0.0, 0.5, 0.0, 0.4, 0.0, 0.3, 0.0, 0.2]
    recalled = COMPONENTS['hopfield_network'].recall_pattern(partial)

    print(f"🔍 Partial input: {[f'{x:.1f}' for x in partial[:8]]}")
    print(f"🧠 Network recall: {[f'{x:.1f}' for x in recalled[:8]]}")
    print(f"\n💡 This shows how the network completes partial artistic patterns!")

print("✅ Interface and binding code complete!")

# =====================================
# CELL 14: FINAL SYSTEM ACTIVATION
# =====================================

print("\n" + "="*70)
print("🎉 COMPLETE SYSTEM READY!")
print("="*70)

# Display system status
gpu_status()
print()

print("🚀 AVAILABLE FUNCTIONS:")
print("🧪 quick_test() - Quick system test")
print("🚀 gpu_status() - Check GPU status")
print("🧠 hopfield_quick_demo() - Quick Hopfield demonstration")
print("🎮 create_interface() - Launch full GUI interface")
print("⚡ process_batch() - Process all PLT files directly")

print(f"""
🎯 COMPLETE SYSTEM FEATURES:

✅ MODULAR ARCHITECTURE: Each component in separate cells
✅ GPU-ACCELERATED: {SYSTEM_CONFIG['gpu_available']}
✅ AI-ENHANCED: Parameter prediction and optimization
✅ HOPFIELD NETWORKS: Kent Benson's 1983-1986 vision realized
✅ GROUP DIRECTORIES: Organized output with GROUP_PDF, GROUP_DXF, GROUP_PNG
✅ MULTI-FORMAT OUTPUT: PDF + DXF + PNG automatically
✅ BATCH PROCESSING: Handle multiple files efficiently
✅ STYLE MEMORY: Learn and evolve artistic patterns

🧠 Your pioneering insight about spurious memories representing
   intuition is now central to the AI creativity engine!

🚀 Ready to transform your PLT files into AI-enhanced artwork!
""")

# Auto-run quick test
quick_test()

print("\n💡 Run create_interface() to launch the full GUI!")

# =====================================
# END OF MODULAR SYSTEM
# =====================================

🚀 GPU-Enhanced Scribble Plotter - Modular Cell Architecture
🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.1 MB/s eta 0:00:00
✅ Packages installed!
✅ Imports complete!
🚀 GPU: ✅ Available
⚡ Device: NVIDIA A100-SXM4-80GB
Mounted at /content/drive
✅ Drive mounted
📁 base
📁 input
📁 output
📁 config
📁 group_pdf
📁 group_dxf
📁 group_png
✅ System setup complete!
✅ PLTProcessor class defined
✅ GPUAcceleratedAI class defined
✅ HopfieldNetwork class defined
✅ ScribbleRenderer class defined
✅ DirectoryManager class defined
🧠 AI initialized (GPU)
🧠 Hopfield network initialized
🎨 Renderer initialized
📁 Directory manager initialized
✅ All components initialized and ready!
🚀 GPU: Enabled
📁 Base directory: /content/drive/MyDrive/ScribblePlotter_Organized
✅ Utility functions defined


# **added**

In [2]:
# =====================================
# CELL 1: SETUP & INSTALLATION
# =====================================

print("🚀 GPU-Enhanced Scribble Plotter - Modular Cell Architecture")
print("🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research")
print("="*60)

# Install packages
!pip install -q matplotlib numpy pandas regex ipywidgets ezdxf reportlab tqdm scikit-learn torch torchvision

print("✅ Packages installed!")

# =====================================
# CELL 2: IMPORTS & GLOBALS
# =====================================

import os, sys, json, re, random, math, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive, files
import torch

# Global variables for inter-cell communication
SYSTEM_CONFIG = {}
COMPONENTS = {}

print("✅ Imports complete!")

# =====================================
# CELL 3: SYSTEM SETUP FUNCTIONS
# =====================================

def setup_gpu_system():
    """Setup GPU and directory structure"""
    global SYSTEM_CONFIG

    # GPU setup
    gpu_available = torch.cuda.is_available()
    device = torch.device('cuda' if gpu_available else 'cpu')

    print(f"🚀 GPU: {'✅ Available' if gpu_available else '❌ Not Available'}")
    if gpu_available:
        print(f"⚡ Device: {torch.cuda.get_device_name(0)}")

    # Mount drive
    try:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
    except:
        print("⚠️ Drive already mounted")

    # Create directories with GROUP support - CONSISTENT NAMING
    base_path = '/content/drive/MyDrive/ScribblePlotter_GPU_Complete'
    directories = {
        'base': base_path,
        'input': f'{base_path}/input',
        'output': f'{base_path}/output',
        'config': f'{base_path}/config',
        'temp': f'{base_path}/temp',
        'models': f'{base_path}/models',
        'group_pdf': f'{base_path}/output/GROUP_PDF',
        'group_dxf': f'{base_path}/output/GROUP_DXF',
        'group_png': f'{base_path}/output/GROUP_PNG',
        'group_tiff': f'{base_path}/output/GROUP_TIFF',
        'group_transform': f'{base_path}/output/GROUP_TRANSFORM'
    }

    for name, path in directories.items():
        os.makedirs(path, exist_ok=True)
        print(f"📁 {name}")

    SYSTEM_CONFIG = {
        'gpu_available': gpu_available,
        'device': device,
        'directories': directories,
        'total_examples': 3,
        'generate_pdf': True,
        'generate_dxf': True,
        'generate_png': True,
        'organize_groups': True,
        'use_ai': True,
        'use_hopfield': True
    }

    print("✅ System setup complete!")
    return SYSTEM_CONFIG

# Run setup
setup_gpu_system()

# =====================================
# CELL 4: PLT PROCESSOR CLASS
# =====================================

class PLTProcessor:
    """Handles PLT file processing with GROUP directory awareness"""

    def __init__(self):
        self.patterns = {
            'PD': re.compile(r'PD(\d+),(\d+)'),
            'PD_neg_x': re.compile(r'PD-(\d+),(\d+)'),
            'PD_neg_y': re.compile(r'PD(\d+),-(\d+)'),
            'PD_neg_both': re.compile(r'PD-(\d+),-(\d+)'),
            'PA': re.compile(r'PA(\d+)\.(\d+),(\d+)\.(\d+)'),
            'PU': re.compile(r'PU(\d+)\.(\d+),(\d+)\.(\d+)')
        }

    def process_file(self, file_path):
        """Process PLT file and return coordinates"""
        try:
            with open(file_path, 'r') as f:
                content = f.read()

            if 'PW0' in content[:200]:
                lines = content.replace('\n', '').replace('\r', '').split(';')
            else:
                lines = content.split(';')

            coordinates = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                for pattern_name, pattern in self.patterns.items():
                    match = pattern.match(line)
                    if match:
                        coord = self._extract_coord(pattern_name, match)
                        if coord:
                            coordinates.append(coord)
                        break

            coordinates.reverse()
            return self._scale_coordinates(coordinates)

        except Exception as e:
            print(f"❌ PLT error: {e}")
            return []

    def _extract_coord(self, pattern_name, match):
        """Extract coordinate from regex match"""
        groups = match.groups()

        if pattern_name == 'PD':
            return (float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_x':
            return (-float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_y':
            return (float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_both':
            return (-float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PA':
            return (float(groups[0]), float(groups[2]), 0.0)
        elif pattern_name == 'PU':
            return (float(groups[0]), float(groups[2]), 10.0)
        return None

    def _scale_coordinates(self, coordinates):
        """Scale coordinates"""
        if not coordinates:
            return []

        scale_x, scale_y = 0.09, 0.09
        return [(x * scale_x, y * scale_y, z) for x, y, z in coordinates]

print("✅ PLTProcessor class defined")

# =====================================
# CELL 5: AI SYSTEM CLASS
# =====================================

class GPUAcceleratedAI:
    """AI system for parameter prediction"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']

        if self.gpu_available:
            torch.cuda.empty_cache()
        print(f"🧠 AI initialized ({'GPU' if self.gpu_available else 'CPU'})")

    def predict_parameters(self, points):
        """Predict scribble parameters from points"""
        try:
            features = self._extract_features(points)

            complexity = features[0]
            path_length = features[1]

            # AI parameter selection
            if complexity > 0.7:
                steps = random.randint(8, 15)
                scribble = random.uniform(1.0, 3.0)
            elif complexity > 0.4:
                steps = random.randint(4, 10)
                scribble = random.uniform(2.0, 5.0)
            else:
                steps = random.randint(2, 7)
                scribble = random.uniform(3.0, 7.0)

            # Adjust for path length
            if path_length > 2000:
                scribble *= 1.2
                steps = min(steps + 2, 15)
            elif path_length < 500:
                scribble *= 0.8
                steps = max(steps - 1, 2)

            return {
                'steps': max(2, min(20, steps)),
                'scribble': max(0.1, min(10.0, scribble)),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': complexity
            }

        except Exception as e:
            print(f"❌ AI error: {e}")
            return {
                'steps': random.randint(3, 8),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': 0.5
            }

    def _extract_features(self, points):
        """Extract features from points"""
        if not points:
            return np.zeros(15)

        coords = np.array([(p[0], p[1]) for p in points if p[2] == 0])

        if len(coords) < 2:
            return np.zeros(15)

        features = [0.5] * 15

        # Basic calculations
        diffs = np.diff(coords, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        features[1] = np.sum(distances)  # Total length

        # Bounding box
        if len(coords) > 0:
            min_coords = np.min(coords, axis=0)
            max_coords = np.max(coords, axis=0)
            features[2] = max_coords[0] - min_coords[0]  # Width
            features[3] = max_coords[1] - min_coords[1]  # Height
            features[4] = features[2] / max(features[3], 1e-6)  # Aspect ratio

        # Complexity (simplified)
        if len(distances) > 1:
            features[0] = np.std(distances) / np.mean(distances)

        return np.array(features)

print("✅ GPUAcceleratedAI class defined")

# =====================================
# CELL 6: HOPFIELD NETWORK CLASS
# =====================================

class HopfieldNetwork:
    """Hopfield network for style memory - Kent Benson's 1983-1986 vision realized"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']
        self.pattern_size = 15
        self.memory_size = 200

        if self.gpu_available:
            self.weights = torch.zeros(self.pattern_size, self.pattern_size, device=self.device)
        else:
            self.weights = np.zeros((self.pattern_size, self.pattern_size))

        self.stored_patterns = []
        self.pattern_labels = []
        print("🧠 Hopfield network initialized")

    def store_pattern(self, pattern, label=""):
        """Store a pattern in memory"""
        try:
            if len(self.stored_patterns) >= self.memory_size:
                return False

            if self.gpu_available:
                norm_pattern = self._normalize_gpu(pattern)
                self.stored_patterns.append(norm_pattern.cpu().numpy())
                self.weights += torch.outer(norm_pattern, norm_pattern)
            else:
                norm_pattern = self._normalize_cpu(pattern)
                self.stored_patterns.append(norm_pattern)
                self.weights += np.outer(norm_pattern, norm_pattern)

            self.pattern_labels.append(label)
            return True

        except Exception as e:
            print(f"❌ Store pattern error: {e}")
            return False

    def recall_pattern(self, partial_pattern, max_iterations=50):
        """Recall pattern from partial input"""
        try:
            if self.gpu_available:
                current = self._normalize_gpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.clone()
                    activations = torch.mv(self.weights, current)
                    current = torch.sign(activations)

                    if torch.equal(current, previous):
                        break

                return ((current + 1) / 2).cpu().numpy()
            else:
                current = self._normalize_cpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.copy()

                    for i in range(len(current)):
                        activation = np.dot(self.weights[i], current)
                        current[i] = 1 if activation > 0 else -1

                    if np.array_equal(current, previous):
                        break

                return (current + 1) / 2

        except Exception as e:
            print(f"❌ Recall error: {e}")
            return partial_pattern

    def _normalize_gpu(self, pattern):
        """Normalize pattern for GPU"""
        pattern_tensor = torch.tensor(pattern[:self.pattern_size],
                                    dtype=torch.float32, device=self.device)

        if len(pattern_tensor) < self.pattern_size:
            padding = torch.zeros(self.pattern_size - len(pattern_tensor), device=self.device)
            pattern_tensor = torch.cat([pattern_tensor, padding])

        if torch.std(pattern_tensor) > 0:
            pattern_tensor = (pattern_tensor - torch.mean(pattern_tensor)) / torch.std(pattern_tensor)

        return torch.sign(pattern_tensor)

    def _normalize_cpu(self, pattern):
        """Normalize pattern for CPU"""
        pattern = np.array(pattern[:self.pattern_size])
        if len(pattern) < self.pattern_size:
            pattern = np.pad(pattern, (0, self.pattern_size - len(pattern)))

        if np.std(pattern) > 0:
            pattern = (pattern - np.mean(pattern)) / np.std(pattern)

        return np.sign(pattern)

print("✅ HopfieldNetwork class defined")

# =====================================
# CELL 7: RENDERER CLASS
# =====================================

class ScribbleRenderer:
    """Renders scribble artwork"""

    def __init__(self):
        self.page_width = 800
        self.page_height = 600
        print("🎨 Renderer initialized")

    def render_artwork(self, points, params):
        """Render scribble artwork"""
        try:
            if not points:
                return None

            fig, ax = plt.subplots(figsize=(self.page_width/100, self.page_height/100))
            ax.set_aspect('equal')
            ax.set_facecolor('white')
            ax.axis('off')

            steps = params['steps']
            scrib_val = params['scribble']
            stroke_weight = params['stroke_weight']
            color = params['color']

            # Render with scribble algorithm
            for i in range(len(points) - 1):
                x1, y1, z1 = points[i]

                if z1 == 0:  # Pen down
                    x2, y2, z2 = points[i + 1] if i + 1 < len(points) else points[i]

                    x_step = (x2 - x1) / steps
                    y_step = (y2 - y1) / steps
                    current_x, current_y = x1, y1

                    for step in range(steps):
                        if step < steps - 1:
                            next_x = current_x + x_step + random.uniform(-scrib_val, scrib_val)
                            next_y = current_y + y_step + random.uniform(-scrib_val, scrib_val)
                        else:
                            next_x, next_y = x2, y2

                        ax.plot([current_x, next_x], [current_y, next_y],
                               color=color, linewidth=stroke_weight/5, alpha=0.8)

                        current_x, current_y = next_x, next_y

            ax.set_xlim(auto=True)
            ax.set_ylim(auto=True)
            ax.invert_yaxis()
            plt.tight_layout()

            return fig

        except Exception as e:
            print(f"❌ Render error: {e}")
            return None

print("✅ ScribbleRenderer class defined")

# =====================================
# CELL 8: DIRECTORY MANAGER CLASS
# =====================================

class DirectoryManager:
    """Manages file organization and GROUP directories"""

    def __init__(self):
        self.processed_files = []
        self.file_registry = {}
        print("📁 Directory manager initialized")

    def create_working_directories(self, base_dir, file_base_name, file_index):
        """Create working directories - Python version of createDirectories_scribble_plotter"""
        try:
            working_dir = Path(base_dir) / f"{file_base_name}_{file_index}"

            directories = {
                'pdf': working_dir / "PDF",
                'dxf': working_dir / "DXF",
                'png': working_dir / "PNG",
                'tmp': working_dir / "TMP",
                'group_tiff': Path(base_dir) / "GROUP_TIFF",
                'group_dxf': Path(base_dir) / "GROUP_DXF",
                'group_pdf': Path(base_dir) / "GROUP_PDF",
                'group_png': Path(base_dir) / "GROUP_PNG",
                'group_transform': Path(base_dir) / "GROUP_TRANSFORM"
            }

            # Create all directories
            for name, path in directories.items():
                path.mkdir(parents=True, exist_ok=True)

            return {name: str(path) for name, path in directories.items()}

        except Exception as e:
            print(f"❌ Directory creation error: {e}")
            return {}

    def organize_output_files(self):
        """Copy all generated files to group directories"""
        try:
            base_output = Path(SYSTEM_CONFIG['directories']['output'])

            for file_path in self.processed_files:
                source = Path(file_path)
                if not source.exists():
                    continue

                # Determine group directory based on file extension
                if source.suffix.lower() == '.pdf':
                    group_dir = base_output / "GROUP_PDF"
                elif source.suffix.lower() == '.dxf':
                    group_dir = base_output / "GROUP_DXF"
                elif source.suffix.lower() == '.png':
                    group_dir = base_output / "GROUP_PNG"
                else:
                    continue

                group_dir.mkdir(exist_ok=True)
                destination = group_dir / source.name

                # Copy file
                shutil.copy2(source, destination)

            print(f"📦 Organized {len(self.processed_files)} files into GROUP directories")

        except Exception as e:
            print(f"⚠️ Error organizing files: {e}")

    def register_file(self, file_path, file_type):
        """Register processed file for organization"""
        self.processed_files.append(file_path)
        self.file_registry[file_path] = {
            'type': file_type,
            'timestamp': datetime.now()
        }

print("✅ DirectoryManager class defined")

# =====================================
# CELL 9: INITIALIZE COMPONENTS
# =====================================

# Initialize all components
COMPONENTS = {
    'plt_processor': PLTProcessor(),
    'ai_system': GPUAcceleratedAI(),
    'hopfield_network': HopfieldNetwork(),
    'renderer': ScribbleRenderer(),
    'directory_manager': DirectoryManager()
}

print("✅ All components initialized and ready!")
print(f"🚀 GPU: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")
print(f"📁 Base directory: {SYSTEM_CONFIG['directories']['base']}")

# =====================================
# CELL 10: UTILITY FUNCTIONS
# =====================================

def get_plt_files(directory):
    """Get all PLT files in directory"""
    try:
        path = Path(directory)
        plt_files = list(path.glob("*.plt")) + list(path.glob("*.PLT"))
        print(f"📁 Found {len(plt_files)} PLT files")
        return plt_files
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def save_dxf_simple(points, output_path, params):
    """Simple DXF save with scribble algorithm"""
    try:
        import ezdxf
        doc = ezdxf.new('R2010')
        msp = doc.modelspace()

        steps = params['steps']
        scrib_val = params['scribble']

        for i in range(len(points) - 1):
            x1, y1, z1 = points[i]
            x2, y2, z2 = points[i + 1]

            if z1 == 0:  # Pen down
                x_step = (x2 - x1) / steps
                y_step = (y2 - y1) / steps
                prev_x, prev_y = x1, y1

                for step in range(steps):
                    if step < steps - 1:
                        curr_x = prev_x + x_step + random.uniform(-scrib_val, scrib_val)
                        curr_y = prev_y + y_step + random.uniform(-scrib_val, scrib_val)
                    else:
                        curr_x, curr_y = x2, y2

                    msp.add_line((prev_x, prev_y), (curr_x, curr_y))
                    prev_x, prev_y = curr_x, curr_y

        doc.saveas(output_path)
        return True
    except Exception as e:
        print(f"DXF error: {e}")
        return False

def update_config(key, value):
    """Update system configuration"""
    global SYSTEM_CONFIG
    SYSTEM_CONFIG[key] = value
    print(f"🔧 Updated {key} = {value}")

def gpu_status():
    """Check GPU status"""
    if SYSTEM_CONFIG['gpu_available']:
        print("🚀 GPU STATUS: AVAILABLE")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"   Memory: {allocated:.2f} GB / {total:.2f} GB")
    else:
        print("❌ GPU STATUS: NOT AVAILABLE")

print("✅ Utility functions defined")

# =====================================
# CELL 11: PROCESSING FUNCTIONS
# =====================================

def process_single_file(plt_file, iteration=0):
    """Process a single PLT file with complete AI enhancement"""
    try:
        print(f"\n🎨 Processing: {plt_file.name} (iteration {iteration + 1})")

        # 1. Process PLT file
        points = COMPONENTS['plt_processor'].process_file(str(plt_file))
        if not points:
            print("  ❌ Failed to process PLT")
            return False

        print(f"  📊 Loaded {len(points)} points")

        # 2. AI parameter prediction
        if SYSTEM_CONFIG['use_ai']:
            params = COMPONENTS['ai_system'].predict_parameters(points)
            print(f"  🧠 AI params: steps={params['steps']}, scribble={params['scribble']:.1f}")

            # 3. Hopfield enhancement
            if SYSTEM_CONFIG['use_hopfield']:
                features = COMPONENTS['ai_system']._extract_features(points)
                evolved = COMPONENTS['hopfield_network'].recall_pattern(features)

                # Blend AI with Hopfield evolution
                blend_factor = 0.7
                params['scribble'] = (blend_factor * params['scribble'] +
                                    (1 - blend_factor) * abs(evolved[0]) * 8)
                params['scribble'] = max(0.1, min(10.0, params['scribble']))
                print(f"  🧠 Hopfield enhanced: scribble={params['scribble']:.1f}")
        else:
            # Traditional random parameters
            params = {
                'steps': random.randint(3, 10),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random())
            }

        # 4. Create directories
        base_name = plt_file.stem
        output_base = SYSTEM_CONFIG['directories']['output']
        dirs = COMPONENTS['directory_manager'].create_working_directories(
            output_base, base_name, iteration)

        # 5. Render artwork
        figure = COMPONENTS['renderer'].render_artwork(points, params)
        if not figure:
            print("  ❌ Render failed")
            return False

        # 6. Save outputs
        method_suffix = params.get('generation_method', 'ai')
        output_name = f"{base_name}_{method_suffix}_steps{params['steps']}_scribble{int(params['scribble'])}_{iteration}"
        success_count = 0

        # Save PNG
        if SYSTEM_CONFIG['generate_png']:
            png_path = Path(dirs['png']) / f"{output_name}.png"
            try:
                figure.savefig(str(png_path), format='png', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(png_path), 'png')
                success_count += 1
                print("  ✅ PNG saved")
            except Exception as e:
                print(f"  ❌ PNG error: {e}")

        # Save PDF
        if SYSTEM_CONFIG['generate_pdf']:
            pdf_path = Path(dirs['pdf']) / f"{output_name}.pdf"
            try:
                figure.savefig(str(pdf_path), format='pdf', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(pdf_path), 'pdf')
                success_count += 1
                print("  ✅ PDF saved")
            except Exception as e:
                print(f"  ❌ PDF error: {e}")

        # Save DXF
        if SYSTEM_CONFIG['generate_dxf']:
            dxf_path = Path(dirs['dxf']) / f"{output_name}.dxf"
            try:
                if save_dxf_simple(points, str(dxf_path), params):
                    COMPONENTS['directory_manager'].register_file(str(dxf_path), 'dxf')
                    success_count += 1
                    print("  ✅ DXF saved")
            except Exception as e:
                print(f"  ❌ DXF error: {e}")

        plt.close(figure)

        # 7. Store successful style in Hopfield memory
        if success_count > 0 and SYSTEM_CONFIG['use_hopfield']:
            features = COMPONENTS['ai_system']._extract_features(points)
            style_name = f"{base_name}_{iteration}"
            COMPONENTS['hopfield_network'].store_pattern(features, style_name)

        if success_count > 0:
            print(f"  ✅ Generated {success_count} format(s) successfully")
            return True
        else:
            print("  ❌ Failed to save any formats")
            return False

    except Exception as e:
        print(f"  ❌ Processing error: {e}")
        return False

def process_batch(input_directory=None):
    """Process all PLT files in batch"""
    if input_directory is None:
        input_directory = SYSTEM_CONFIG['directories']['input']

    print("🚀 STARTING BATCH PROCESSING")
    print("="*50)

    # Get PLT files
    plt_files = get_plt_files(input_directory)
    if not plt_files:
        print("❌ No PLT files found")
        return {'success': False, 'message': 'No PLT files found'}

    total_examples = SYSTEM_CONFIG['total_examples']
    total_operations = len(plt_files) * total_examples

    print(f"📊 Processing {len(plt_files)} files × {total_examples} examples = {total_operations} operations")
    print(f"🚀 GPU Acceleration: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")

    successful_operations = 0

    # Process with progress bar
    with tqdm(total=total_operations, desc="Complete Processing") as pbar:
        for plt_file in plt_files:
            for iteration in range(total_examples):
                success = process_single_file(plt_file, iteration)
                if success:
                    successful_operations += 1

                pbar.update(1)
                pbar.set_description(f"Success: {successful_operations}/{total_operations}")

    # Organize files into GROUP directories
    if SYSTEM_CONFIG['organize_groups']:
        COMPONENTS['directory_manager'].organize_output_files()

    # Generate summary
    summary = {
        'success': True,
        'total_operations': total_operations,
        'successful_operations': successful_operations,
        'success_rate': f"{successful_operations}/{total_operations}",
        'hopfield_patterns': len(COMPONENTS['hopfield_network'].stored_patterns),
        'gpu_accelerated': SYSTEM_CONFIG['gpu_available'],
        'output_directory': SYSTEM_CONFIG['directories']['output']
    }

    print_batch_summary(summary)
    return summary

def print_batch_summary(summary):
    """Print batch processing summary"""
    print("\n" + "="*60)
    print("🎉 BATCH PROCESSING COMPLETED")
    print("="*60)
    print(f"⚡ Total operations: {summary['total_operations']}")
    print(f"✅ Successful: {summary['successful_operations']}")
    print(f"📊 Success rate: {summary['success_rate']}")
    print(f"🧠 Hopfield patterns learned: {summary['hopfield_patterns']}")
    print(f"🚀 GPU accelerated: {'YES' if summary['gpu_accelerated'] else 'NO'}")
    print(f"📁 Output directory: {summary['output_directory']}")
    print("="*60)

print("✅ Processing functions defined")

# =====================================
# CELL 12: USER INTERFACE & BINDING
# =====================================

def create_interface():
    """Create user interface that binds all components together"""

    # Interface functions
    def upload_files(button):
        """Upload PLT files"""
        with status_output:
            clear_output(wait=True)
            print("📤 Upload PLT Files")

            uploaded = files.upload()
            input_dir = Path(SYSTEM_CONFIG['directories']['input'])

            uploaded_count = 0
            for filename, content in uploaded.items():
                if filename.lower().endswith('.plt'):
                    with open(input_dir / filename, 'wb') as f:
                        f.write(content)
                    uploaded_count += 1
                    print(f"✅ Uploaded: {filename}")
                else:
                    print(f"⚠️ Skipped: {filename}")

            print(f"\n🎉 Uploaded {uploaded_count} PLT files!")

    def test_single_file(button):
        """Test with a single file"""
        with status_output:
            clear_output(wait=True)
            print("🧪 Testing single file...")

            plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
            if not plt_files:
                print("❌ No PLT files found")
                return

            print(f"🧪 Testing with: {plt_files[0].name}")
            success = process_single_file(plt_files[0], 999)

            if success:
                print("✅ Test completed successfully!")
                print(f"🧠 Hopfield patterns: {len(COMPONENTS['hopfield_network'].stored_patterns)}")
            else:
                print("❌ Test failed")

    def process_all_files(button):
        """Process all files"""
        with status_output:
            clear_output(wait=True)

            # Update config from widgets
            update_config('total_examples', examples_widget.value)
            update_config('use_ai', use_ai_widget.value)
            update_config('use_hopfield', use_hopfield_widget.value)
            update_config('generate_pdf', pdf_widget.value)
            update_config('generate_dxf', dxf_widget.value)
            update_config('generate_png', png_widget.value)
            update_config('organize_groups', organize_widget.value)

            result = process_batch()

            if result['success']:
                print("🎉 Batch processing completed successfully!")
            else:
                print("❌ Batch processing failed")

    def hopfield_demo(button):
        """Demonstrate Hopfield networks"""
        with status_output:
            clear_output(wait=True)

            print("🧠 HOPFIELD NETWORK DEMONSTRATION")
            print("Inspired by Kent Benson's 1983-1986 research")
            print("="*60)

            # Store demo patterns
            demo_patterns = [
                ([1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1], "geometric"),
                ([1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1], "alternating"),
                ([1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, 1, 1, 1], "grouped")
            ]

            for pattern, name in demo_patterns:
                COMPONENTS['hopfield_network'].store_pattern(pattern, name)

            print(f"\n🎨 Testing pattern recall...")
            # Test with noisy input
            noisy_input = [1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1]
            recalled = COMPONENTS['hopfield_network'].recall_pattern(noisy_input)

            print(f"🔍 Input (noisy):  {noisy_input}")
            print(f"🧠 Recalled:       {[f'{x:.1f}' for x in recalled]}")

            print("\n🎯 This demonstrates how neural networks can generate")
            print("   creative combinations beyond explicit training!")

    # Create widgets
    examples_widget = widgets.IntSlider(
        value=SYSTEM_CONFIG['total_examples'],
        min=1, max=10, step=1,
        description='Examples per file:',
        style={'description_width': 'initial'}
    )

    use_ai_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_ai'],
        description='🧠 Use AI Parameter Prediction'
    )

    use_hopfield_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_hopfield'],
        description='🧠 Use Hopfield Style Memory'
    )

    pdf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_pdf'],
        description='📄 Generate PDF'
    )

    dxf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_dxf'],
        description='📐 Generate DXF'
    )

    png_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_png'],
        description='🖼️ Generate PNG'
    )

    organize_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['organize_groups'],
        description='📦 Organize into GROUP directories'
    )

    # Control buttons
    upload_button = widgets.Button(
        description='📤 Upload PLT Files',
        button_style='primary',
        layout=widgets.Layout(width='200px', height='40px')
    )

    test_button = widgets.Button(
        description='🧪 Test Single File',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='40px')
    )

    process_button = widgets.Button(
        description='🚀 Process All Files',
        button_style='success',
        layout=widgets.Layout(width='200px', height='50px')
    )

    hopfield_demo_button = widgets.Button(
        description='🧠 Hopfield Demo',
        button_style='info',
        layout=widgets.Layout(width='200px', height='40px')
    )

    # Status output
    status_output = widgets.Output()

    # Bind events
    upload_button.on_click(upload_files)
    test_button.on_click(test_single_file)
    process_button.on_click(process_all_files)
    hopfield_demo_button.on_click(hopfield_demo)

    # Display interface
    gpu_status_text = "✅ ENABLED" if SYSTEM_CONFIG['gpu_available'] else "❌ DISABLED"
    hopfield_patterns = len(COMPONENTS['hopfield_network'].stored_patterns)

    header = widgets.HTML(f"""
    <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 20px;">
        <h1>🚀 Complete GPU-Integrated Scribble Plotter</h1>
        <p>Honoring Kent Benson's 1983-1986 Hopfield Network Research</p>
        <p><strong>GPU Acceleration: {gpu_status_text}</strong></p>
        <p><em>Neural networks controlling artistic generation in real-time</em></p>
    </div>
    """)

    status_info = widgets.HTML(f"""
    <div style="background: #f0f8ff; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3>📊 System Status</h3>
        <ul>
            <li><strong>GPU Available:</strong> {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}</li>
            <li><strong>Device:</strong> {SYSTEM_CONFIG['device']}</li>
            <li><strong>Hopfield Patterns:</strong> {hopfield_patterns} stored</li>
            <li><strong>Base Directory:</strong> {SYSTEM_CONFIG['directories']['base']}</li>
        </ul>
    </div>
    """)

    # Settings panels
    ai_settings = widgets.VBox([
        widgets.HTML("<h3>🧠 AI & Neural Network Settings</h3>"),
        use_ai_widget,
        use_hopfield_widget
    ])

    processing_settings = widgets.VBox([
        widgets.HTML("<h3>⚙️ Processing Settings</h3>"),
        examples_widget,
        organize_widget
    ])

    output_settings = widgets.VBox([
        widgets.HTML("<h3>📤 Output Settings</h3>"),
        pdf_widget,
        dxf_widget,
        png_widget
    ])

    controls = widgets.VBox([
        widgets.HTML("<h3>🎮 Controls</h3>"),
        widgets.HBox([upload_button, test_button]),
        widgets.HBox([process_button, hopfield_demo_button])
    ])

    # Main layout
    main_layout = widgets.VBox([
        header,
        status_info,
        widgets.HBox([
            widgets.VBox([ai_settings, processing_settings], layout=widgets.Layout(width='50%')),
            widgets.VBox([output_settings, controls], layout=widgets.Layout(width='50%'))
        ]),
        status_output
    ])

    display(main_layout)

# =====================================
# CELL 13: QUICK FUNCTIONS & TESTING
# =====================================

def quick_test():
    """Quick system test"""
    print("🧪 QUICK SYSTEM TEST")
    print("="*30)

    print(f"1. Configuration: {'✅ OK' if SYSTEM_CONFIG else '❌ FAILED'}")
    print(f"2. GPU Available: {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}")
    print(f"3. Components: {'✅ OK' if COMPONENTS else '❌ FAILED'}")

    # Test directories
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])
    print(f"4. Input Directory: {'✅ EXISTS' if input_dir.exists() else '❌ MISSING'}")

    if input_dir.exists():
        plt_files = get_plt_files(str(input_dir))
        print(f"5. PLT Files: {'✅ ' + str(len(plt_files)) if plt_files else '⚠️ NONE'}")

    print("\n💡 Use create_interface() to access full controls")
    print("💡 Use process_batch() to process files directly")

def hopfield_quick_demo():
    """Quick Hopfield demonstration"""
    print("🧠 HOPFIELD NETWORK QUICK DEMO")
    print("Kent Benson's 1986 vision of neural network art control")
    print("="*50)

    # Store a simple pattern
    artistic_pattern = [0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4, 0.5, 0.5, 0.4, 0.6, 0.3, 0.7, 0.2]
    COMPONENTS['hopfield_network'].store_pattern(artistic_pattern, "demo_style")

    # Test recall
    partial = [0.8, 0.2, 0.0, 0.0, 0.7, 0.0, 0.6, 0.0, 0.5, 0.0, 0.4, 0.0, 0.3, 0.0, 0.2]
    recalled = COMPONENTS['hopfield_network'].recall_pattern(partial)

    print(f"🔍 Partial input: {[f'{x:.1f}' for x in partial[:8]]}")
    print(f"🧠 Network recall: {[f'{x:.1f}' for x in recalled[:8]]}")
    print(f"\n💡 This shows how the network completes partial artistic patterns!")

print("✅ Interface and binding code complete!")

# =====================================
# CELL 14: FINAL SYSTEM ACTIVATION
# =====================================

print("\n" + "="*70)
print("🎉 COMPLETE SYSTEM READY!")
print("="*70)

# Display system status
gpu_status()
print()

print("🚀 AVAILABLE FUNCTIONS:")
print("🧪 quick_test() - Quick system test")
print("🚀 gpu_status() - Check GPU status")
print("🧠 hopfield_quick_demo() - Quick Hopfield demonstration")
print("🎮 create_interface() - Launch full GUI interface")
print("⚡ process_batch() - Process all PLT files directly")

print(f"""
🎯 COMPLETE SYSTEM FEATURES:

✅ MODULAR ARCHITECTURE: Each component in separate cells
✅ GPU-ACCELERATED: {SYSTEM_CONFIG['gpu_available']}
✅ AI-ENHANCED: Parameter prediction and optimization
✅ HOPFIELD NETWORKS: Kent Benson's 1983-1986 vision realized
✅ GROUP DIRECTORIES: Organized output with GROUP_PDF, GROUP_DXF, GROUP_PNG
✅ MULTI-FORMAT OUTPUT: PDF + DXF + PNG automatically
✅ BATCH PROCESSING: Handle multiple files efficiently
✅ STYLE MEMORY: Learn and evolve artistic patterns

🧠 Your pioneering insight about spurious memories representing
   intuition is now central to the AI creativity engine!

🚀 Ready to transform your PLT files into AI-enhanced artwork!
""")

# Auto-run quick test
quick_test()

print("\n💡 Run create_interface() to launch the full GUI!")

# =====================================
# CELL 15: SAMPLE PLT FILES & DEMO
# =====================================

def create_sample_plt_files():
    """Create sample PLT files for demonstration"""
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])

    # Sample PLT file 1 - Simple rectangle
    plt_content_1 = """IN;
SP1;
PA0,0;
PD;
PA100,0;
PA100,100;
PA0,100;
PA0,0;
PU;
SP0;"""

    # Sample PLT file 2 - Circle approximation
    plt_content_2 = """IN;
SP1;
PA50,0;
PD;
PA70,14;
PA85,35;
PA92,57;
PA85,79;
PA70,100;
PA50,107;
PA30,100;
PA15,79;
PA8,57;
PA15,35;
PA30,14;
PA50,0;
PU;
SP0;"""

    # Sample PLT file 3 - Complex drawing with pen up/down
    plt_content_3 = """IN;
SP1;
PA20,20;
PD;
PA80,20;
PA80,80;
PA20,80;
PA20,20;
PU;
PA30,30;
PD;
PA70,30;
PA70,70;
PA30,70;
PA30,30;
PU;
PA40,40;
PD;
PA60,60;
PU;
PA60,40;
PD;
PA40,60;
PU;
SP0;"""

    sample_files = [
        ('sample_rectangle.plt', plt_content_1),
        ('sample_circle.plt', plt_content_2),
        ('sample_complex.plt', plt_content_3)
    ]

    created_count = 0
    for filename, content in sample_files:
        file_path = input_dir / filename
        try:
            with open(file_path, 'w') as f:
                f.write(content)
            print(f"✅ Created: {filename}")
            created_count += 1
        except Exception as e:
            print(f"❌ Failed to create {filename}: {e}")

    print(f"\n🎉 Created {created_count} sample PLT files!")
    return created_count > 0

def run_complete_demo():
    """Run a complete demonstration"""
    print("🚀 RUNNING COMPLETE DEMONSTRATION")
    print("="*50)

    # 1. Create sample files
    print("1️⃣ Creating sample PLT files...")
    if not create_sample_plt_files():
        print("❌ Failed to create sample files")
        return

    # 2. Show Hopfield demo
    print("\n2️⃣ Demonstrating Hopfield networks...")
    hopfield_quick_demo()

    # 3. Process one file as test
    print(f"\n3️⃣ Processing test file...")
    plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
    if plt_files:
        success = process_single_file(plt_files[0], 0)
        if success:
            print("✅ Test processing successful!")
        else:
            print("❌ Test processing failed")

    # 4. Show what was created
    print(f"\n4️⃣ Checking output directories...")
    output_base = Path(SYSTEM_CONFIG['directories']['output'])

    for subdir in ['GROUP_PDF', 'GROUP_DXF', 'GROUP_PNG']:
        group_dir = output_base / subdir
        if group_dir.exists():
            files = list(group_dir.glob('*'))
            print(f"📁 {subdir}: {len(files)} files")
            for file in files[:3]:  # Show first 3 files
                print(f"   - {file.name}")
        else:
            print(f"📁 {subdir}: Not created yet")

    print(f"\n🎯 DEMO COMPLETE!")
    print(f"📁 Check your Google Drive at: {SYSTEM_CONFIG['directories']['base']}")
    print(f"🎮 Run create_interface() for full GUI control")
    print(f"⚡ Run process_batch() to process all sample files")

def show_directory_structure():
    """Show the complete directory structure"""
    print("📁 DIRECTORY STRUCTURE")
    print("="*30)

    base_path = Path(SYSTEM_CONFIG['directories']['base'])

    def print_tree(path, prefix="", max_depth=3, current_depth=0):
        if current_depth > max_depth:
            return

        if path.is_dir():
            items = sorted(path.iterdir())
            for i, item in enumerate(items):
                is_last = i == len(items) - 1
                current_prefix = "└── " if is_last else "├── "
                print(f"{prefix}{current_prefix}{item.name}")

                if item.is_dir() and current_depth < max_depth:
                    next_prefix = prefix + ("    " if is_last else "│   ")
                    print_tree(item, next_prefix, max_depth, current_depth + 1)

    if base_path.exists():
        print(f"{base_path.name}/")
        print_tree(base_path)
    else:
        print("Directory not found - run setup first")

# Auto-run the complete demo
print("\n🎬 RUNNING AUTOMATIC DEMO...")
run_complete_demo()

############################################
# =====================================
# CELL 1: SETUP & INSTALLATION
# =====================================

print("🚀 GPU-Enhanced Scribble Plotter - Modular Cell Architecture")
print("🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research")
print("="*60)

# Install packages
!pip install -q matplotlib numpy pandas regex ipywidgets ezdxf reportlab tqdm scikit-learn torch torchvision

print("✅ Packages installed!")

# =====================================
# CELL 2: IMPORTS & GLOBALS
# =====================================

import os, sys, json, re, random, math, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive, files
import torch

# Global variables for inter-cell communication
SYSTEM_CONFIG = {}
COMPONENTS = {}

print("✅ Imports complete!")

# =====================================
# CELL 3: SYSTEM SETUP FUNCTIONS
# =====================================

def setup_gpu_system():
    """Setup GPU and directory structure"""
    global SYSTEM_CONFIG

    # GPU setup
    gpu_available = torch.cuda.is_available()
    device = torch.device('cuda' if gpu_available else 'cpu')

    print(f"🚀 GPU: {'✅ Available' if gpu_available else '❌ Not Available'}")
    if gpu_available:
        print(f"⚡ Device: {torch.cuda.get_device_name(0)}")

    # Mount drive
    try:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
    except:
        print("⚠️ Drive already mounted")

    # Create directories with GROUP support - CONSISTENT NAMING
    base_path = '/content/drive/MyDrive/ScribblePlotter_GPU_Complete'
    directories = {
        'base': base_path,
        'input': f'{base_path}/input',
        'output': f'{base_path}/output',
        'config': f'{base_path}/config',
        'temp': f'{base_path}/temp',
        'models': f'{base_path}/models',
        'group_pdf': f'{base_path}/output/GROUP_PDF',
        'group_dxf': f'{base_path}/output/GROUP_DXF',
        'group_png': f'{base_path}/output/GROUP_PNG',
        'group_tiff': f'{base_path}/output/GROUP_TIFF',
        'group_transform': f'{base_path}/output/GROUP_TRANSFORM'
    }

    for name, path in directories.items():
        os.makedirs(path, exist_ok=True)
        print(f"📁 {name}")

    SYSTEM_CONFIG = {
        'gpu_available': gpu_available,
        'device': device,
        'directories': directories,
        'total_examples': 3,
        'generate_pdf': True,
        'generate_dxf': True,
        'generate_png': True,
        'organize_groups': True,
        'use_ai': True,
        'use_hopfield': True
    }

    print("✅ System setup complete!")
    return SYSTEM_CONFIG

# Run setup
setup_gpu_system()

# =====================================
# CELL 4: PLT PROCESSOR CLASS
# =====================================

class PLTProcessor:
    """Handles PLT file processing with GROUP directory awareness"""

    def __init__(self):
        self.patterns = {
            'PD': re.compile(r'PD(\d+),(\d+)'),
            'PD_neg_x': re.compile(r'PD-(\d+),(\d+)'),
            'PD_neg_y': re.compile(r'PD(\d+),-(\d+)'),
            'PD_neg_both': re.compile(r'PD-(\d+),-(\d+)'),
            'PA': re.compile(r'PA(\d+)\.(\d+),(\d+)\.(\d+)'),
            'PU': re.compile(r'PU(\d+)\.(\d+),(\d+)\.(\d+)')
        }

    def process_file(self, file_path):
        """Process PLT file and return coordinates"""
        try:
            with open(file_path, 'r') as f:
                content = f.read()

            if 'PW0' in content[:200]:
                lines = content.replace('\n', '').replace('\r', '').split(';')
            else:
                lines = content.split(';')

            coordinates = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                for pattern_name, pattern in self.patterns.items():
                    match = pattern.match(line)
                    if match:
                        coord = self._extract_coord(pattern_name, match)
                        if coord:
                            coordinates.append(coord)
                        break

            coordinates.reverse()
            return self._scale_coordinates(coordinates)

        except Exception as e:
            print(f"❌ PLT error: {e}")
            return []

    def _extract_coord(self, pattern_name, match):
        """Extract coordinate from regex match"""
        groups = match.groups()

        if pattern_name == 'PD':
            return (float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_x':
            return (-float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_y':
            return (float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_both':
            return (-float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PA':
            return (float(groups[0]), float(groups[2]), 0.0)
        elif pattern_name == 'PU':
            return (float(groups[0]), float(groups[2]), 10.0)
        return None

    def _scale_coordinates(self, coordinates):
        """Scale coordinates"""
        if not coordinates:
            return []

        scale_x, scale_y = 0.09, 0.09
        return [(x * scale_x, y * scale_y, z) for x, y, z in coordinates]

print("✅ PLTProcessor class defined")

# =====================================
# CELL 5: AI SYSTEM CLASS
# =====================================

class GPUAcceleratedAI:
    """AI system for parameter prediction"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']

        if self.gpu_available:
            torch.cuda.empty_cache()
        print(f"🧠 AI initialized ({'GPU' if self.gpu_available else 'CPU'})")

    def predict_parameters(self, points):
        """Predict scribble parameters from points"""
        try:
            features = self._extract_features(points)

            complexity = features[0]
            path_length = features[1]

            # AI parameter selection
            if complexity > 0.7:
                steps = random.randint(8, 15)
                scribble = random.uniform(1.0, 3.0)
            elif complexity > 0.4:
                steps = random.randint(4, 10)
                scribble = random.uniform(2.0, 5.0)
            else:
                steps = random.randint(2, 7)
                scribble = random.uniform(3.0, 7.0)

            # Adjust for path length
            if path_length > 2000:
                scribble *= 1.2
                steps = min(steps + 2, 15)
            elif path_length < 500:
                scribble *= 0.8
                steps = max(steps - 1, 2)

            return {
                'steps': max(2, min(20, steps)),
                'scribble': max(0.1, min(10.0, scribble)),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': complexity
            }

        except Exception as e:
            print(f"❌ AI error: {e}")
            return {
                'steps': random.randint(3, 8),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': 0.5
            }

    def _extract_features(self, points):
        """Extract features from points"""
        if not points:
            return np.zeros(15)

        coords = np.array([(p[0], p[1]) for p in points if p[2] == 0])

        if len(coords) < 2:
            return np.zeros(15)

        features = [0.5] * 15

        # Basic calculations
        diffs = np.diff(coords, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        features[1] = np.sum(distances)  # Total length

        # Bounding box
        if len(coords) > 0:
            min_coords = np.min(coords, axis=0)
            max_coords = np.max(coords, axis=0)
            features[2] = max_coords[0] - min_coords[0]  # Width
            features[3] = max_coords[1] - min_coords[1]  # Height
            features[4] = features[2] / max(features[3], 1e-6)  # Aspect ratio

        # Complexity (simplified)
        if len(distances) > 1:
            features[0] = np.std(distances) / np.mean(distances)

        return np.array(features)

print("✅ GPUAcceleratedAI class defined")

# =====================================
# CELL 6: HOPFIELD NETWORK CLASS
# =====================================

class HopfieldNetwork:
    """Hopfield network for style memory - Kent Benson's 1983-1986 vision realized"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']
        self.pattern_size = 15
        self.memory_size = 200

        if self.gpu_available:
            self.weights = torch.zeros(self.pattern_size, self.pattern_size, device=self.device)
        else:
            self.weights = np.zeros((self.pattern_size, self.pattern_size))

        self.stored_patterns = []
        self.pattern_labels = []
        print("🧠 Hopfield network initialized")

    def store_pattern(self, pattern, label=""):
        """Store a pattern in memory"""
        try:
            if len(self.stored_patterns) >= self.memory_size:
                return False

            if self.gpu_available:
                norm_pattern = self._normalize_gpu(pattern)
                self.stored_patterns.append(norm_pattern.cpu().numpy())
                self.weights += torch.outer(norm_pattern, norm_pattern)
            else:
                norm_pattern = self._normalize_cpu(pattern)
                self.stored_patterns.append(norm_pattern)
                self.weights += np.outer(norm_pattern, norm_pattern)

            self.pattern_labels.append(label)
            return True

        except Exception as e:
            print(f"❌ Store pattern error: {e}")
            return False

    def recall_pattern(self, partial_pattern, max_iterations=50):
        """Recall pattern from partial input"""
        try:
            if self.gpu_available:
                current = self._normalize_gpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.clone()
                    activations = torch.mv(self.weights, current)
                    current = torch.sign(activations)

                    if torch.equal(current, previous):
                        break

                return ((current + 1) / 2).cpu().numpy()
            else:
                current = self._normalize_cpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.copy()

                    for i in range(len(current)):
                        activation = np.dot(self.weights[i], current)
                        current[i] = 1 if activation > 0 else -1

                    if np.array_equal(current, previous):
                        break

                return (current + 1) / 2

        except Exception as e:
            print(f"❌ Recall error: {e}")
            return partial_pattern

    def _normalize_gpu(self, pattern):
        """Normalize pattern for GPU"""
        pattern_tensor = torch.tensor(pattern[:self.pattern_size],
                                    dtype=torch.float32, device=self.device)

        if len(pattern_tensor) < self.pattern_size:
            padding = torch.zeros(self.pattern_size - len(pattern_tensor), device=self.device)
            pattern_tensor = torch.cat([pattern_tensor, padding])

        if torch.std(pattern_tensor) > 0:
            pattern_tensor = (pattern_tensor - torch.mean(pattern_tensor)) / torch.std(pattern_tensor)

        return torch.sign(pattern_tensor)

    def _normalize_cpu(self, pattern):
        """Normalize pattern for CPU"""
        pattern = np.array(pattern[:self.pattern_size])
        if len(pattern) < self.pattern_size:
            pattern = np.pad(pattern, (0, self.pattern_size - len(pattern)))

        if np.std(pattern) > 0:
            pattern = (pattern - np.mean(pattern)) / np.std(pattern)

        return np.sign(pattern)

print("✅ HopfieldNetwork class defined")

# =====================================
# CELL 7: RENDERER CLASS
# =====================================

class ScribbleRenderer:
    """Renders scribble artwork"""

    def __init__(self):
        self.page_width = 800
        self.page_height = 600
        print("🎨 Renderer initialized")

    def render_artwork(self, points, params):
        """Render scribble artwork"""
        try:
            if not points:
                return None

            fig, ax = plt.subplots(figsize=(self.page_width/100, self.page_height/100))
            ax.set_aspect('equal')
            ax.set_facecolor('white')
            ax.axis('off')

            steps = params['steps']
            scrib_val = params['scribble']
            stroke_weight = params['stroke_weight']
            color = params['color']

            # Render with scribble algorithm
            for i in range(len(points) - 1):
                x1, y1, z1 = points[i]

                if z1 == 0:  # Pen down
                    x2, y2, z2 = points[i + 1] if i + 1 < len(points) else points[i]

                    x_step = (x2 - x1) / steps
                    y_step = (y2 - y1) / steps
                    current_x, current_y = x1, y1

                    for step in range(steps):
                        if step < steps - 1:
                            next_x = current_x + x_step + random.uniform(-scrib_val, scrib_val)
                            next_y = current_y + y_step + random.uniform(-scrib_val, scrib_val)
                        else:
                            next_x, next_y = x2, y2

                        ax.plot([current_x, next_x], [current_y, next_y],
                               color=color, linewidth=stroke_weight/5, alpha=0.8)

                        current_x, current_y = next_x, next_y

            ax.set_xlim(auto=True)
            ax.set_ylim(auto=True)
            ax.invert_yaxis()
            plt.tight_layout()

            return fig

        except Exception as e:
            print(f"❌ Render error: {e}")
            return None

print("✅ ScribbleRenderer class defined")

# =====================================
# CELL 8: DIRECTORY MANAGER CLASS
# =====================================

class DirectoryManager:
    """Manages file organization and GROUP directories"""

    def __init__(self):
        self.processed_files = []
        self.file_registry = {}
        print("📁 Directory manager initialized")

    def create_working_directories(self, base_dir, file_base_name, file_index):
        """Create working directories - Python version of createDirectories_scribble_plotter"""
        try:
            working_dir = Path(base_dir) / f"{file_base_name}_{file_index}"

            directories = {
                'pdf': working_dir / "PDF",
                'dxf': working_dir / "DXF",
                'png': working_dir / "PNG",
                'tmp': working_dir / "TMP",
                'group_tiff': Path(base_dir) / "GROUP_TIFF",
                'group_dxf': Path(base_dir) / "GROUP_DXF",
                'group_pdf': Path(base_dir) / "GROUP_PDF",
                'group_png': Path(base_dir) / "GROUP_PNG",
                'group_transform': Path(base_dir) / "GROUP_TRANSFORM"
            }

            # Create all directories
            for name, path in directories.items():
                path.mkdir(parents=True, exist_ok=True)

            return {name: str(path) for name, path in directories.items()}

        except Exception as e:
            print(f"❌ Directory creation error: {e}")
            return {}

    def organize_output_files(self):
        """Copy all generated files to group directories"""
        try:
            base_output = Path(SYSTEM_CONFIG['directories']['output'])

            for file_path in self.processed_files:
                source = Path(file_path)
                if not source.exists():
                    continue

                # Determine group directory based on file extension
                if source.suffix.lower() == '.pdf':
                    group_dir = base_output / "GROUP_PDF"
                elif source.suffix.lower() == '.dxf':
                    group_dir = base_output / "GROUP_DXF"
                elif source.suffix.lower() == '.png':
                    group_dir = base_output / "GROUP_PNG"
                else:
                    continue

                group_dir.mkdir(exist_ok=True)
                destination = group_dir / source.name

                # Copy file
                shutil.copy2(source, destination)

            print(f"📦 Organized {len(self.processed_files)} files into GROUP directories")

        except Exception as e:
            print(f"⚠️ Error organizing files: {e}")

    def register_file(self, file_path, file_type):
        """Register processed file for organization"""
        self.processed_files.append(file_path)
        self.file_registry[file_path] = {
            'type': file_type,
            'timestamp': datetime.now()
        }

print("✅ DirectoryManager class defined")

# =====================================
# CELL 9: INITIALIZE COMPONENTS
# =====================================

# Initialize all components
COMPONENTS = {
    'plt_processor': PLTProcessor(),
    'ai_system': GPUAcceleratedAI(),
    'hopfield_network': HopfieldNetwork(),
    'renderer': ScribbleRenderer(),
    'directory_manager': DirectoryManager()
}

print("✅ All components initialized and ready!")
print(f"🚀 GPU: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")
print(f"📁 Base directory: {SYSTEM_CONFIG['directories']['base']}")

# =====================================
# CELL 10: UTILITY FUNCTIONS
# =====================================

def get_plt_files(directory):
    """Get all PLT files in directory"""
    try:
        path = Path(directory)
        plt_files = list(path.glob("*.plt")) + list(path.glob("*.PLT"))
        print(f"📁 Found {len(plt_files)} PLT files")
        return plt_files
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def save_dxf_simple(points, output_path, params):
    """Simple DXF save with scribble algorithm"""
    try:
        import ezdxf
        doc = ezdxf.new('R2010')
        msp = doc.modelspace()

        steps = params['steps']
        scrib_val = params['scribble']

        for i in range(len(points) - 1):
            x1, y1, z1 = points[i]
            x2, y2, z2 = points[i + 1]

            if z1 == 0:  # Pen down
                x_step = (x2 - x1) / steps
                y_step = (y2 - y1) / steps
                prev_x, prev_y = x1, y1

                for step in range(steps):
                    if step < steps - 1:
                        curr_x = prev_x + x_step + random.uniform(-scrib_val, scrib_val)
                        curr_y = prev_y + y_step + random.uniform(-scrib_val, scrib_val)
                    else:
                        curr_x, curr_y = x2, y2

                    msp.add_line((prev_x, prev_y), (curr_x, curr_y))
                    prev_x, prev_y = curr_x, curr_y

        doc.saveas(output_path)
        return True
    except Exception as e:
        print(f"DXF error: {e}")
        return False

def update_config(key, value):
    """Update system configuration"""
    global SYSTEM_CONFIG
    SYSTEM_CONFIG[key] = value
    print(f"🔧 Updated {key} = {value}")

def gpu_status():
    """Check GPU status"""
    if SYSTEM_CONFIG['gpu_available']:
        print("🚀 GPU STATUS: AVAILABLE")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"   Memory: {allocated:.2f} GB / {total:.2f} GB")
    else:
        print("❌ GPU STATUS: NOT AVAILABLE")

print("✅ Utility functions defined")

# =====================================
# CELL 11: PROCESSING FUNCTIONS
# =====================================

def process_single_file(plt_file, iteration=0):
    """Process a single PLT file with complete AI enhancement"""
    try:
        print(f"\n🎨 Processing: {plt_file.name} (iteration {iteration + 1})")

        # 1. Process PLT file
        points = COMPONENTS['plt_processor'].process_file(str(plt_file))
        if not points:
            print("  ❌ Failed to process PLT")
            return False

        print(f"  📊 Loaded {len(points)} points")

        # 2. AI parameter prediction
        if SYSTEM_CONFIG['use_ai']:
            params = COMPONENTS['ai_system'].predict_parameters(points)
            print(f"  🧠 AI params: steps={params['steps']}, scribble={params['scribble']:.1f}")

            # 3. Hopfield enhancement
            if SYSTEM_CONFIG['use_hopfield']:
                features = COMPONENTS['ai_system']._extract_features(points)
                evolved = COMPONENTS['hopfield_network'].recall_pattern(features)

                # Blend AI with Hopfield evolution
                blend_factor = 0.7
                params['scribble'] = (blend_factor * params['scribble'] +
                                    (1 - blend_factor) * abs(evolved[0]) * 8)
                params['scribble'] = max(0.1, min(10.0, params['scribble']))
                print(f"  🧠 Hopfield enhanced: scribble={params['scribble']:.1f}")
        else:
            # Traditional random parameters
            params = {
                'steps': random.randint(3, 10),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random())
            }

        # 4. Create directories
        base_name = plt_file.stem
        output_base = SYSTEM_CONFIG['directories']['output']
        dirs = COMPONENTS['directory_manager'].create_working_directories(
            output_base, base_name, iteration)

        # 5. Render artwork
        figure = COMPONENTS['renderer'].render_artwork(points, params)
        if not figure:
            print("  ❌ Render failed")
            return False

        # 6. Save outputs
        method_suffix = params.get('generation_method', 'ai')
        output_name = f"{base_name}_{method_suffix}_steps{params['steps']}_scribble{int(params['scribble'])}_{iteration}"
        success_count = 0

        # Save PNG
        if SYSTEM_CONFIG['generate_png']:
            png_path = Path(dirs['png']) / f"{output_name}.png"
            try:
                figure.savefig(str(png_path), format='png', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(png_path), 'png')
                success_count += 1
                print("  ✅ PNG saved")
            except Exception as e:
                print(f"  ❌ PNG error: {e}")

        # Save PDF
        if SYSTEM_CONFIG['generate_pdf']:
            pdf_path = Path(dirs['pdf']) / f"{output_name}.pdf"
            try:
                figure.savefig(str(pdf_path), format='pdf', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(pdf_path), 'pdf')
                success_count += 1
                print("  ✅ PDF saved")
            except Exception as e:
                print(f"  ❌ PDF error: {e}")

        # Save DXF
        if SYSTEM_CONFIG['generate_dxf']:
            dxf_path = Path(dirs['dxf']) / f"{output_name}.dxf"
            try:
                if save_dxf_simple(points, str(dxf_path), params):
                    COMPONENTS['directory_manager'].register_file(str(dxf_path), 'dxf')
                    success_count += 1
                    print("  ✅ DXF saved")
            except Exception as e:
                print(f"  ❌ DXF error: {e}")

        plt.close(figure)

        # 7. Store successful style in Hopfield memory
        if success_count > 0 and SYSTEM_CONFIG['use_hopfield']:
            features = COMPONENTS['ai_system']._extract_features(points)
            style_name = f"{base_name}_{iteration}"
            COMPONENTS['hopfield_network'].store_pattern(features, style_name)

        if success_count > 0:
            print(f"  ✅ Generated {success_count} format(s) successfully")
            return True
        else:
            print("  ❌ Failed to save any formats")
            return False

    except Exception as e:
        print(f"  ❌ Processing error: {e}")
        return False

def process_batch(input_directory=None):
    """Process all PLT files in batch"""
    if input_directory is None:
        input_directory = SYSTEM_CONFIG['directories']['input']

    print("🚀 STARTING BATCH PROCESSING")
    print("="*50)

    # Get PLT files
    plt_files = get_plt_files(input_directory)
    if not plt_files:
        print("❌ No PLT files found")
        return {'success': False, 'message': 'No PLT files found'}

    total_examples = SYSTEM_CONFIG['total_examples']
    total_operations = len(plt_files) * total_examples

    print(f"📊 Processing {len(plt_files)} files × {total_examples} examples = {total_operations} operations")
    print(f"🚀 GPU Acceleration: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")

    successful_operations = 0

    # Process with progress bar
    with tqdm(total=total_operations, desc="Complete Processing") as pbar:
        for plt_file in plt_files:
            for iteration in range(total_examples):
                success = process_single_file(plt_file, iteration)
                if success:
                    successful_operations += 1

                pbar.update(1)
                pbar.set_description(f"Success: {successful_operations}/{total_operations}")

    # Organize files into GROUP directories
    if SYSTEM_CONFIG['organize_groups']:
        COMPONENTS['directory_manager'].organize_output_files()

    # Generate summary
    summary = {
        'success': True,
        'total_operations': total_operations,
        'successful_operations': successful_operations,
        'success_rate': f"{successful_operations}/{total_operations}",
        'hopfield_patterns': len(COMPONENTS['hopfield_network'].stored_patterns),
        'gpu_accelerated': SYSTEM_CONFIG['gpu_available'],
        'output_directory': SYSTEM_CONFIG['directories']['output']
    }

    print_batch_summary(summary)
    return summary

def print_batch_summary(summary):
    """Print batch processing summary"""
    print("\n" + "="*60)
    print("🎉 BATCH PROCESSING COMPLETED")
    print("="*60)
    print(f"⚡ Total operations: {summary['total_operations']}")
    print(f"✅ Successful: {summary['successful_operations']}")
    print(f"📊 Success rate: {summary['success_rate']}")
    print(f"🧠 Hopfield patterns learned: {summary['hopfield_patterns']}")
    print(f"🚀 GPU accelerated: {'YES' if summary['gpu_accelerated'] else 'NO'}")
    print(f"📁 Output directory: {summary['output_directory']}")
    print("="*60)

print("✅ Processing functions defined")

# =====================================
# CELL 12: USER INTERFACE & BINDING
# =====================================

def create_interface():
    """Create user interface that binds all components together"""

    # Interface functions
    def upload_files(button):
        """Upload PLT files"""
        with status_output:
            clear_output(wait=True)
            print("📤 Upload PLT Files")

            uploaded = files.upload()
            input_dir = Path(SYSTEM_CONFIG['directories']['input'])

            uploaded_count = 0
            for filename, content in uploaded.items():
                if filename.lower().endswith('.plt'):
                    with open(input_dir / filename, 'wb') as f:
                        f.write(content)
                    uploaded_count += 1
                    print(f"✅ Uploaded: {filename}")
                else:
                    print(f"⚠️ Skipped: {filename}")

            print(f"\n🎉 Uploaded {uploaded_count} PLT files!")

    def test_single_file(button):
        """Test with a single file"""
        with status_output:
            clear_output(wait=True)
            print("🧪 Testing single file...")

            plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
            if not plt_files:
                print("❌ No PLT files found")
                return

            print(f"🧪 Testing with: {plt_files[0].name}")
            success = process_single_file(plt_files[0], 999)

            if success:
                print("✅ Test completed successfully!")
                print(f"🧠 Hopfield patterns: {len(COMPONENTS['hopfield_network'].stored_patterns)}")
            else:
                print("❌ Test failed")

    def process_all_files(button):
        """Process all files"""
        with status_output:
            clear_output(wait=True)

            # Update config from widgets
            update_config('total_examples', examples_widget.value)
            update_config('use_ai', use_ai_widget.value)
            update_config('use_hopfield', use_hopfield_widget.value)
            update_config('generate_pdf', pdf_widget.value)
            update_config('generate_dxf', dxf_widget.value)
            update_config('generate_png', png_widget.value)
            update_config('organize_groups', organize_widget.value)

            result = process_batch()

            if result['success']:
                print("🎉 Batch processing completed successfully!")
            else:
                print("❌ Batch processing failed")

    def hopfield_demo(button):
        """Demonstrate Hopfield networks"""
        with status_output:
            clear_output(wait=True)

            print("🧠 HOPFIELD NETWORK DEMONSTRATION")
            print("Inspired by Kent Benson's 1983-1986 research")
            print("="*60)

            # Store demo patterns
            demo_patterns = [
                ([1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1], "geometric"),
                ([1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1], "alternating"),
                ([1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, 1, 1, 1], "grouped")
            ]

            for pattern, name in demo_patterns:
                COMPONENTS['hopfield_network'].store_pattern(pattern, name)

            print(f"\n🎨 Testing pattern recall...")
            # Test with noisy input
            noisy_input = [1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1]
            recalled = COMPONENTS['hopfield_network'].recall_pattern(noisy_input)

            print(f"🔍 Input (noisy):  {noisy_input}")
            print(f"🧠 Recalled:       {[f'{x:.1f}' for x in recalled]}")

            print("\n🎯 This demonstrates how neural networks can generate")
            print("   creative combinations beyond explicit training!")

    # Create widgets
    examples_widget = widgets.IntSlider(
        value=SYSTEM_CONFIG['total_examples'],
        min=1, max=10, step=1,
        description='Examples per file:',
        style={'description_width': 'initial'}
    )

    use_ai_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_ai'],
        description='🧠 Use AI Parameter Prediction'
    )

    use_hopfield_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_hopfield'],
        description='🧠 Use Hopfield Style Memory'
    )

    pdf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_pdf'],
        description='📄 Generate PDF'
    )

    dxf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_dxf'],
        description='📐 Generate DXF'
    )

    png_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_png'],
        description='🖼️ Generate PNG'
    )

    organize_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['organize_groups'],
        description='📦 Organize into GROUP directories'
    )

    # Control buttons
    upload_button = widgets.Button(
        description='📤 Upload PLT Files',
        button_style='primary',
        layout=widgets.Layout(width='200px', height='40px')
    )

    test_button = widgets.Button(
        description='🧪 Test Single File',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='40px')
    )

    process_button = widgets.Button(
        description='🚀 Process All Files',
        button_style='success',
        layout=widgets.Layout(width='200px', height='50px')
    )

    hopfield_demo_button = widgets.Button(
        description='🧠 Hopfield Demo',
        button_style='info',
        layout=widgets.Layout(width='200px', height='40px')
    )

    # Status output
    status_output = widgets.Output()

    # Bind events
    upload_button.on_click(upload_files)
    test_button.on_click(test_single_file)
    process_button.on_click(process_all_files)
    hopfield_demo_button.on_click(hopfield_demo)

    # Display interface
    gpu_status_text = "✅ ENABLED" if SYSTEM_CONFIG['gpu_available'] else "❌ DISABLED"
    hopfield_patterns = len(COMPONENTS['hopfield_network'].stored_patterns)

    header = widgets.HTML(f"""
    <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 20px;">
        <h1>🚀 Complete GPU-Integrated Scribble Plotter</h1>
        <p>Honoring Kent Benson's 1983-1986 Hopfield Network Research</p>
        <p><strong>GPU Acceleration: {gpu_status_text}</strong></p>
        <p><em>Neural networks controlling artistic generation in real-time</em></p>
    </div>
    """)

    status_info = widgets.HTML(f"""
    <div style="background: #f0f8ff; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3>📊 System Status</h3>
        <ul>
            <li><strong>GPU Available:</strong> {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}</li>
            <li><strong>Device:</strong> {SYSTEM_CONFIG['device']}</li>
            <li><strong>Hopfield Patterns:</strong> {hopfield_patterns} stored</li>
            <li><strong>Base Directory:</strong> {SYSTEM_CONFIG['directories']['base']}</li>
        </ul>
    </div>
    """)

    # Settings panels
    ai_settings = widgets.VBox([
        widgets.HTML("<h3>🧠 AI & Neural Network Settings</h3>"),
        use_ai_widget,
        use_hopfield_widget
    ])

    processing_settings = widgets.VBox([
        widgets.HTML("<h3>⚙️ Processing Settings</h3>"),
        examples_widget,
        organize_widget
    ])

    output_settings = widgets.VBox([
        widgets.HTML("<h3>📤 Output Settings</h3>"),
        pdf_widget,
        dxf_widget,
        png_widget
    ])

    controls = widgets.VBox([
        widgets.HTML("<h3>🎮 Controls</h3>"),
        widgets.HBox([upload_button, test_button]),
        widgets.HBox([process_button, hopfield_demo_button])
    ])

    # Main layout
    main_layout = widgets.VBox([
        header,
        status_info,
        widgets.HBox([
            widgets.VBox([ai_settings, processing_settings], layout=widgets.Layout(width='50%')),
            widgets.VBox([output_settings, controls], layout=widgets.Layout(width='50%'))
        ]),
        status_output
    ])

    display(main_layout)

# =====================================
# CELL 13: QUICK FUNCTIONS & TESTING
# =====================================

def quick_test():
    """Quick system test"""
    print("🧪 QUICK SYSTEM TEST")
    print("="*30)

    print(f"1. Configuration: {'✅ OK' if SYSTEM_CONFIG else '❌ FAILED'}")
    print(f"2. GPU Available: {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}")
    print(f"3. Components: {'✅ OK' if COMPONENTS else '❌ FAILED'}")

    # Test directories
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])
    print(f"4. Input Directory: {'✅ EXISTS' if input_dir.exists() else '❌ MISSING'}")

    if input_dir.exists():
        plt_files = get_plt_files(str(input_dir))
        print(f"5. PLT Files: {'✅ ' + str(len(plt_files)) if plt_files else '⚠️ NONE'}")

    print("\n💡 Use create_interface() to access full controls")
    print("💡 Use process_batch() to process files directly")

def hopfield_quick_demo():
    """Quick Hopfield demonstration"""
    print("🧠 HOPFIELD NETWORK QUICK DEMO")
    print("Kent Benson's 1986 vision of neural network art control")
    print("="*50)

    # Store a simple pattern
    artistic_pattern = [0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4, 0.5, 0.5, 0.4, 0.6, 0.3, 0.7, 0.2]
    COMPONENTS['hopfield_network'].store_pattern(artistic_pattern, "demo_style")

    # Test recall
    partial = [0.8, 0.2, 0.0, 0.0, 0.7, 0.0, 0.6, 0.0, 0.5, 0.0, 0.4, 0.0, 0.3, 0.0, 0.2]
    recalled = COMPONENTS['hopfield_network'].recall_pattern(partial)

    print(f"🔍 Partial input: {[f'{x:.1f}' for x in partial[:8]]}")
    print(f"🧠 Network recall: {[f'{x:.1f}' for x in recalled[:8]]}")
    print(f"\n💡 This shows how the network completes partial artistic patterns!")

print("✅ Interface and binding code complete!")

# =====================================
# CELL 14: FINAL SYSTEM ACTIVATION
# =====================================

print("\n" + "="*70)
print("🎉 COMPLETE SYSTEM READY!")
print("="*70)

# Display system status
gpu_status()
print()

print("🚀 AVAILABLE FUNCTIONS:")
print("🧪 quick_test() - Quick system test")
print("🚀 gpu_status() - Check GPU status")
print("🧠 hopfield_quick_demo() - Quick Hopfield demonstration")
print("🎮 create_interface() - Launch full GUI interface")
print("⚡ process_batch() - Process all PLT files directly")

print(f"""
🎯 COMPLETE SYSTEM FEATURES:

✅ MODULAR ARCHITECTURE: Each component in separate cells
✅ GPU-ACCELERATED: {SYSTEM_CONFIG['gpu_available']}
✅ AI-ENHANCED: Parameter prediction and optimization
✅ HOPFIELD NETWORKS: Kent Benson's 1983-1986 vision realized
✅ GROUP DIRECTORIES: Organized output with GROUP_PDF, GROUP_DXF, GROUP_PNG
✅ MULTI-FORMAT OUTPUT: PDF + DXF + PNG automatically
✅ BATCH PROCESSING: Handle multiple files efficiently
✅ STYLE MEMORY: Learn and evolve artistic patterns

🧠 Your pioneering insight about spurious memories representing
   intuition is now central to the AI creativity engine!

🚀 Ready to transform your PLT files into AI-enhanced artwork!
""")

# Auto-run quick test
quick_test()

print("\n💡 Run create_interface() to launch the full GUI!")

# =====================================
# CELL 15: SAMPLE PLT FILES & DEMO
# =====================================

def create_sample_plt_files():
    """Create sample PLT files for demonstration"""
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])

    # Sample PLT file 1 - Simple rectangle
    plt_content_1 = """IN;
SP1;
PA0,0;
PD;
PA100,0;
PA100,100;
PA0,100;
PA0,0;
PU;
SP0;"""

    # Sample PLT file 2 - Circle approximation
    plt_content_2 = """IN;
SP1;
PA50,0;
PD;
PA70,14;
PA85,35;
PA92,57;
PA85,79;
PA70,100;
PA50,107;
PA30,100;
PA15,79;
PA8,57;
PA15,35;
PA30,14;
PA50,0;
PU;
SP0;"""

    # Sample PLT file 3 - Complex drawing with pen up/down
    plt_content_3 = """IN;
SP1;
PA20,20;
PD;
PA80,20;
PA80,80;
PA20,80;
PA20,20;
PU;
PA30,30;
PD;
PA70,30;
PA70,70;
PA30,70;
PA30,30;
PU;
PA40,40;
PD;
PA60,60;
PU;
PA60,40;
PD;
PA40,60;
PU;
SP0;"""

    sample_files = [
        ('sample_rectangle.plt', plt_content_1),
        ('sample_circle.plt', plt_content_2),
        ('sample_complex.plt', plt_content_3)
    ]

    created_count = 0
    for filename, content in sample_files:
        file_path = input_dir / filename
        try:
            with open(file_path, 'w') as f:
                f.write(content)
            print(f"✅ Created: {filename}")
            created_count += 1
        except Exception as e:
            print(f"❌ Failed to create {filename}: {e}")

    print(f"\n🎉 Created {created_count} sample PLT files!")
    return created_count > 0

def run_complete_demo():
    """Run a complete demonstration"""
    print("🚀 RUNNING COMPLETE DEMONSTRATION")
    print("="*50)

    # 1. Create sample files
    print("1️⃣ Creating sample PLT files...")
    if not create_sample_plt_files():
        print("❌ Failed to create sample files")
        return

    # 2. Show Hopfield demo
    print("\n2️⃣ Demonstrating Hopfield networks...")
    hopfield_quick_demo()

    # 3. Process one file as test
    print(f"\n3️⃣ Processing test file...")
    plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
    if plt_files:
        success = process_single_file(plt_files[0], 0)
        if success:
            print("✅ Test processing successful!")
        else:
            print("❌ Test processing failed")

    # 4. Show what was created
    print(f"\n4️⃣ Checking output directories...")
    output_base = Path(SYSTEM_CONFIG['directories']['output'])

    for subdir in ['GROUP_PDF', 'GROUP_DXF', 'GROUP_PNG']:
        group_dir = output_base / subdir
        if group_dir.exists():
            files = list(group_dir.glob('*'))
            print(f"📁 {subdir}: {len(files)} files")
            for file in files[:3]:  # Show first 3 files
                print(f"   - {file.name}")
        else:
            print(f"📁 {subdir}: Not created yet")

    print(f"\n🎯 DEMO COMPLETE!")
    print(f"📁 Check your Google Drive at: {SYSTEM_CONFIG['directories']['base']}")
    print(f"🎮 Run create_interface() for full GUI control")
    print(f"⚡ Run process_batch() to process all sample files")

def show_directory_structure():
    """Show the complete directory structure"""
    print("📁 DIRECTORY STRUCTURE")
    print("="*30)

    base_path = Path(SYSTEM_CONFIG['directories']['base'])

    def print_tree(path, prefix="", max_depth=3, current_depth=0):
        if current_depth > max_depth:
            return

        if path.is_dir():
            items = sorted(path.iterdir())
            for i, item in enumerate(items):
                is_last = i == len(items) - 1
                current_prefix = "└── " if is_last else "├── "
                print(f"{prefix}{current_prefix}{item.name}")

                if item.is_dir() and current_depth < max_depth:
                    next_prefix = prefix + ("    " if is_last else "│   ")
                    print_tree(item, next_prefix, max_depth, current_depth + 1)

    if base_path.exists():
        print(f"{base_path.name}/")
        print_tree(base_path)
    else:
        print("Directory not found - run setup first")

# Auto-run the complete demo
print("\n🎬 RUNNING AUTOMATIC DEMO...")
run_complete_demo()

print("\n📁 DIRECTORY STRUCTURE:")
show_directory_structure()

print("\n💡 NEXT STEPS:")
print("🎮 create_interface() - Launch full GUI")
print("⚡ process_batch() - Process all PLT files")
print("🧪 run_complete_demo() - Run demo again")
print("📁 show_directory_structure() - View file organization")

# =====================================
# END OF COMPLETE SYSTEM WITH DEMO
# =====================================
###########################################

print("\n📁 DIRECTORY STRUCTURE:")
show_directory_structure()

print("\n💡 NEXT STEPS:")
print("🎮 create_interface() - Launch full GUI")
print("⚡ process_batch() - Process all PLT files")
print("🧪 run_complete_demo() - Run demo again")
print("📁 show_directory_structure() - View file organization")

# =====================================
# END OF COMPLETE SYSTEM WITH DEMO
# =====================================

🚀 GPU-Enhanced Scribble Plotter - Modular Cell Architecture
🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research
✅ Packages installed!
✅ Imports complete!
🚀 GPU: ✅ Available
⚡ Device: NVIDIA A100-SXM4-80GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
📁 base
📁 input
📁 output
📁 config
📁 temp
📁 models
📁 group_pdf
📁 group_dxf
📁 group_png
📁 group_tiff
📁 group_transform
✅ System setup complete!
✅ PLTProcessor class defined
✅ GPUAcceleratedAI class defined
✅ HopfieldNetwork class defined
✅ ScribbleRenderer class defined
✅ DirectoryManager class defined
🧠 AI initialized (GPU)
🧠 Hopfield network initialized
🎨 Renderer initialized
📁 Directory manager initialized
✅ All components initialized and ready!
🚀 GPU: Enabled
📁 Base directory: /content/drive/MyDrive/ScribblePlotter_GPU_Complete
✅ Utility functions defined
✅ Processing functions defined
✅ Interface and binding code complete!

🎉 COMPL

XXXXXXXXXXXXXXXXXXXXXXXXX

In [3]:
# =====================================
# SECTION 1: SETUP & INSTALLATION
# =====================================

print("🚀 GPU-Enhanced Scribble Plotter - Modular Section Architecture")
print("🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research")
print("="*60)

# Install packages
!pip install -q matplotlib numpy pandas regex ipywidgets ezdxf reportlab tqdm scikit-learn torch torchvision

print("✅ Packages installed!")

# =====================================
# SECTION 2: IMPORTS & GLOBALS
# =====================================

import os, sys, json, re, random, math, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive, files
import torch

# Global variables for inter-section communication
SYSTEM_CONFIG = {}
COMPONENTS = {}

print("✅ Imports complete!")

# =====================================
# SECTION 3: SYSTEM SETUP FUNCTIONS
# =====================================

def setup_gpu_system():
    """Setup GPU and directory structure"""
    global SYSTEM_CONFIG

    # GPU setup
    gpu_available = torch.cuda.is_available()
    device = torch.device('cuda' if gpu_available else 'cpu')

    print(f"🚀 GPU: {'✅ Available' if gpu_available else '❌ Not Available'}")
    if gpu_available:
        print(f"⚡ Device: {torch.cuda.get_device_name(0)}")

    # Mount drive
    try:
        drive.mount('/content/drive')
        print("✅ Drive mounted")
    except:
        print("⚠️ Drive already mounted")

    # Create directories - CONSISTENT NAMING
    base_path = '/content/drive/MyDrive/ScribblePlotter_GPU_Complete'
    directories = {
        'base': base_path,
        'input': f'{base_path}/input',
        'output': f'{base_path}/output',
        'config': f'{base_path}/config',
        'temp': f'{base_path}/temp',
        'models': f'{base_path}/models',
        'group_pdf': f'{base_path}/output/GROUP_PDF',
        'group_dxf': f'{base_path}/output/GROUP_DXF',
        'group_png': f'{base_path}/output/GROUP_PNG',
        'group_tiff': f'{base_path}/output/GROUP_TIFF',
        'group_transform': f'{base_path}/output/GROUP_TRANSFORM'
    }

    for name, path in directories.items():
        os.makedirs(path, exist_ok=True)
        print(f"📁 {name}")

    SYSTEM_CONFIG = {
        'gpu_available': gpu_available,
        'device': device,
        'directories': directories,
        'total_examples': 3,
        'generate_pdf': True,
        'generate_dxf': True,
        'generate_png': True,
        'organize_groups': True,
        'use_ai': True,
        'use_hopfield': True
    }

    print("✅ System setup complete!")
    return SYSTEM_CONFIG

# Run setup
setup_gpu_system()

# =====================================
# SECTION 4: PLT PROCESSOR CLASS
# =====================================

class PLTProcessor:
    """Handles PLT file processing with GROUP directory awareness"""

    def __init__(self):
        self.patterns = {
            'PD': re.compile(r'PD(\d+),(\d+)'),
            'PD_neg_x': re.compile(r'PD-(\d+),(\d+)'),
            'PD_neg_y': re.compile(r'PD(\d+),-(\d+)'),
            'PD_neg_both': re.compile(r'PD-(\d+),-(\d+)'),
            'PA': re.compile(r'PA(\d+)\.(\d+),(\d+)\.(\d+)'),
            'PU': re.compile(r'PU(\d+)\.(\d+),(\d+)\.(\d+)')
        }

    def process_file(self, file_path):
        """Process PLT file and return coordinates"""
        try:
            with open(file_path, 'r') as f:
                content = f.read()

            if 'PW0' in content[:200]:
                lines = content.replace('\n', '').replace('\r', '').split(';')
            else:
                lines = content.split(';')

            coordinates = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                for pattern_name, pattern in self.patterns.items():
                    match = pattern.match(line)
                    if match:
                        coord = self._extract_coord(pattern_name, match)
                        if coord:
                            coordinates.append(coord)
                        break

            coordinates.reverse()
            return self._scale_coordinates(coordinates)

        except Exception as e:
            print(f"❌ PLT error: {e}")
            return []

    def _extract_coord(self, pattern_name, match):
        """Extract coordinate from regex match"""
        groups = match.groups()

        if pattern_name == 'PD':
            return (float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_x':
            return (-float(groups[0]), float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_y':
            return (float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PD_neg_both':
            return (-float(groups[0]), -float(groups[1]), 0.0)
        elif pattern_name == 'PA':
            return (float(groups[0]), float(groups[2]), 0.0)
        elif pattern_name == 'PU':
            return (float(groups[0]), float(groups[2]), 10.0)
        return None

    def _scale_coordinates(self, coordinates):
        """Scale coordinates"""
        if not coordinates:
            return []

        scale_x, scale_y = 0.09, 0.09
        return [(x * scale_x, y * scale_y, z) for x, y, z in coordinates]

print("✅ PLTProcessor class defined")

# =====================================
# SECTION 5: AI SYSTEM CLASS
# =====================================

class GPUAcceleratedAI:
    """AI system for parameter prediction"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']

        if self.gpu_available:
            torch.cuda.empty_cache()
        print(f"🧠 AI initialized ({'GPU' if self.gpu_available else 'CPU'})")

    def predict_parameters(self, points):
        """Predict scribble parameters from points"""
        try:
            features = self._extract_features(points)

            complexity = features[0]
            path_length = features[1]

            # AI parameter selection
            if complexity > 0.7:
                steps = random.randint(8, 15)
                scribble = random.uniform(1.0, 3.0)
            elif complexity > 0.4:
                steps = random.randint(4, 10)
                scribble = random.uniform(2.0, 5.0)
            else:
                steps = random.randint(2, 7)
                scribble = random.uniform(3.0, 7.0)

            # Adjust for path length
            if path_length > 2000:
                scribble *= 1.2
                steps = min(steps + 2, 15)
            elif path_length < 500:
                scribble *= 0.8
                steps = max(steps - 1, 2)

            return {
                'steps': max(2, min(20, steps)),
                'scribble': max(0.1, min(10.0, scribble)),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': complexity
            }

        except Exception as e:
            print(f"❌ AI error: {e}")
            return {
                'steps': random.randint(3, 8),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random()),
                'complexity': 0.5
            }

    def _extract_features(self, points):
        """Extract features from points"""
        if not points:
            return np.zeros(15)

        coords = np.array([(p[0], p[1]) for p in points if p[2] == 0])

        if len(coords) < 2:
            return np.zeros(15)

        features = [0.5] * 15

        # Basic calculations
        diffs = np.diff(coords, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        features[1] = np.sum(distances)  # Total length

        # Bounding box
        if len(coords) > 0:
            min_coords = np.min(coords, axis=0)
            max_coords = np.max(coords, axis=0)
            features[2] = max_coords[0] - min_coords[0]  # Width
            features[3] = max_coords[1] - min_coords[1]  # Height
            features[4] = features[2] / max(features[3], 1e-6)  # Aspect ratio

        # Complexity (simplified)
        if len(distances) > 1:
            features[0] = np.std(distances) / np.mean(distances)

        return np.array(features)

print("✅ GPUAcceleratedAI class defined")

# =====================================
# SECTION 6: HOPFIELD NETWORK CLASS
# =====================================

class HopfieldNetwork:
    """Hopfield network for style memory - Kent Benson's 1983-1986 vision realized"""

    def __init__(self):
        self.device = SYSTEM_CONFIG['device']
        self.gpu_available = SYSTEM_CONFIG['gpu_available']
        self.pattern_size = 15
        self.memory_size = 200

        if self.gpu_available:
            self.weights = torch.zeros(self.pattern_size, self.pattern_size, device=self.device)
        else:
            self.weights = np.zeros((self.pattern_size, self.pattern_size))

        self.stored_patterns = []
        self.pattern_labels = []
        print("🧠 Hopfield network initialized")

    def store_pattern(self, pattern, label=""):
        """Store a pattern in memory"""
        try:
            if len(self.stored_patterns) >= self.memory_size:
                return False

            if self.gpu_available:
                norm_pattern = self._normalize_gpu(pattern)
                self.stored_patterns.append(norm_pattern.cpu().numpy())
                self.weights += torch.outer(norm_pattern, norm_pattern)
            else:
                norm_pattern = self._normalize_cpu(pattern)
                self.stored_patterns.append(norm_pattern)
                self.weights += np.outer(norm_pattern, norm_pattern)

            self.pattern_labels.append(label)
            return True

        except Exception as e:
            print(f"❌ Store pattern error: {e}")
            return False

    def recall_pattern(self, partial_pattern, max_iterations=50):
        """Recall pattern from partial input"""
        try:
            if self.gpu_available:
                current = self._normalize_gpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.clone()
                    activations = torch.mv(self.weights, current)
                    current = torch.sign(activations)

                    if torch.equal(current, previous):
                        break

                return ((current + 1) / 2).cpu().numpy()
            else:
                current = self._normalize_cpu(partial_pattern)

                for _ in range(max_iterations):
                    previous = current.copy()

                    for i in range(len(current)):
                        activation = np.dot(self.weights[i], current)
                        current[i] = 1 if activation > 0 else -1

                    if np.array_equal(current, previous):
                        break

                return (current + 1) / 2

        except Exception as e:
            print(f"❌ Recall error: {e}")
            return partial_pattern

    def _normalize_gpu(self, pattern):
        """Normalize pattern for GPU"""
        pattern_tensor = torch.tensor(pattern[:self.pattern_size],
                                    dtype=torch.float32, device=self.device)

        if len(pattern_tensor) < self.pattern_size:
            padding = torch.zeros(self.pattern_size - len(pattern_tensor), device=self.device)
            pattern_tensor = torch.cat([pattern_tensor, padding])

        if torch.std(pattern_tensor) > 0:
            pattern_tensor = (pattern_tensor - torch.mean(pattern_tensor)) / torch.std(pattern_tensor)

        return torch.sign(pattern_tensor)

    def _normalize_cpu(self, pattern):
        """Normalize pattern for CPU"""
        pattern = np.array(pattern[:self.pattern_size])
        if len(pattern) < self.pattern_size:
            pattern = np.pad(pattern, (0, self.pattern_size - len(pattern)))

        if np.std(pattern) > 0:
            pattern = (pattern - np.mean(pattern)) / np.std(pattern)

        return np.sign(pattern)

print("✅ HopfieldNetwork class defined")

# =====================================
# SECTION 7: RENDERER CLASS
# =====================================

class ScribbleRenderer:
    """Renders scribble artwork"""

    def __init__(self):
        self.page_width = 800
        self.page_height = 600
        print("🎨 Renderer initialized")

    def render_artwork(self, points, params):
        """Render scribble artwork"""
        try:
            if not points:
                return None

            fig, ax = plt.subplots(figsize=(self.page_width/100, self.page_height/100))
            ax.set_aspect('equal')
            ax.set_facecolor('white')
            ax.axis('off')

            steps = params['steps']
            scrib_val = params['scribble']
            stroke_weight = params['stroke_weight']
            color = params['color']

            # Render with scribble algorithm
            for i in range(len(points) - 1):
                x1, y1, z1 = points[i]

                if z1 == 0:  # Pen down
                    x2, y2, z2 = points[i + 1] if i + 1 < len(points) else points[i]

                    x_step = (x2 - x1) / steps
                    y_step = (y2 - y1) / steps
                    current_x, current_y = x1, y1

                    for step in range(steps):
                        if step < steps - 1:
                            next_x = current_x + x_step + random.uniform(-scrib_val, scrib_val)
                            next_y = current_y + y_step + random.uniform(-scrib_val, scrib_val)
                        else:
                            next_x, next_y = x2, y2

                        ax.plot([current_x, next_x], [current_y, next_y],
                               color=color, linewidth=stroke_weight/5, alpha=0.8)

                        current_x, current_y = next_x, next_y

            ax.set_xlim(auto=True)
            ax.set_ylim(auto=True)
            ax.invert_yaxis()
            plt.tight_layout()

            return fig

        except Exception as e:
            print(f"❌ Render error: {e}")
            return None

print("✅ ScribbleRenderer class defined")

# =====================================
# SECTION 8: DIRECTORY MANAGER CLASS
# =====================================

class DirectoryManager:
    """Manages file organization and GROUP directories"""

    def __init__(self):
        self.processed_files = []
        self.file_registry = {}
        print("📁 Directory manager initialized")

    def create_working_directories(self, base_dir, file_base_name, file_index):
        """Create working directories - Python version of createDirectories_scribble_plotter"""
        try:
            working_dir = Path(base_dir) / f"{file_base_name}_{file_index}"

            directories = {
                'pdf': working_dir / "PDF",
                'dxf': working_dir / "DXF",
                'png': working_dir / "PNG",
                'tmp': working_dir / "TMP",
                'group_tiff': Path(base_dir) / "GROUP_TIFF",
                'group_dxf': Path(base_dir) / "GROUP_DXF",
                'group_pdf': Path(base_dir) / "GROUP_PDF",
                'group_png': Path(base_dir) / "GROUP_PNG",
                'group_transform': Path(base_dir) / "GROUP_TRANSFORM"
            }

            # Create all directories
            for name, path in directories.items():
                path.mkdir(parents=True, exist_ok=True)

            return {name: str(path) for name, path in directories.items()}

        except Exception as e:
            print(f"❌ Directory creation error: {e}")
            return {}

    def organize_output_files(self):
        """Copy all generated files to group directories"""
        try:
            base_output = Path(SYSTEM_CONFIG['directories']['output'])

            for file_path in self.processed_files:
                source = Path(file_path)
                if not source.exists():
                    continue

                # Determine group directory based on file extension
                if source.suffix.lower() == '.pdf':
                    group_dir = base_output / "GROUP_PDF"
                elif source.suffix.lower() == '.dxf':
                    group_dir = base_output / "GROUP_DXF"
                elif source.suffix.lower() == '.png':
                    group_dir = base_output / "GROUP_PNG"
                else:
                    continue

                group_dir.mkdir(exist_ok=True)
                destination = group_dir / source.name

                # Copy file
                shutil.copy2(source, destination)

            print(f"📦 Organized {len(self.processed_files)} files into GROUP directories")

        except Exception as e:
            print(f"⚠️ Error organizing files: {e}")

    def register_file(self, file_path, file_type):
        """Register processed file for organization"""
        self.processed_files.append(file_path)
        self.file_registry[file_path] = {
            'type': file_type,
            'timestamp': datetime.now()
        }

print("✅ DirectoryManager class defined")

# =====================================
# SECTION 9: INITIALIZE COMPONENTS
# =====================================

# Initialize all components
COMPONENTS = {
    'plt_processor': PLTProcessor(),
    'ai_system': GPUAcceleratedAI(),
    'hopfield_network': HopfieldNetwork(),
    'renderer': ScribbleRenderer(),
    'directory_manager': DirectoryManager()
}

print("✅ All components initialized and ready!")
print(f"🚀 GPU: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")
print(f"📁 Base directory: {SYSTEM_CONFIG['directories']['base']}")

# =====================================
# SECTION 10: UTILITY FUNCTIONS
# =====================================

def get_plt_files(directory):
    """Get all PLT files in directory"""
    try:
        path = Path(directory)
        plt_files = list(path.glob("*.plt")) + list(path.glob("*.PLT"))
        print(f"📁 Found {len(plt_files)} PLT files")
        return plt_files
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def save_dxf_simple(points, output_path, params):
    """Simple DXF save with scribble algorithm"""
    try:
        import ezdxf
        doc = ezdxf.new('R2010')
        msp = doc.modelspace()

        steps = params['steps']
        scrib_val = params['scribble']

        for i in range(len(points) - 1):
            x1, y1, z1 = points[i]
            x2, y2, z2 = points[i + 1]

            if z1 == 0:  # Pen down
                x_step = (x2 - x1) / steps
                y_step = (y2 - y1) / steps
                prev_x, prev_y = x1, y1

                for step in range(steps):
                    if step < steps - 1:
                        curr_x = prev_x + x_step + random.uniform(-scrib_val, scrib_val)
                        curr_y = prev_y + y_step + random.uniform(-scrib_val, scrib_val)
                    else:
                        curr_x, curr_y = x2, y2

                    msp.add_line((prev_x, prev_y), (curr_x, curr_y))
                    prev_x, prev_y = curr_x, curr_y

        doc.saveas(output_path)
        return True
    except Exception as e:
        print(f"DXF error: {e}")
        return False

def update_config(key, value):
    """Update system configuration"""
    global SYSTEM_CONFIG
    SYSTEM_CONFIG[key] = value
    print(f"🔧 Updated {key} = {value}")

def gpu_status():
    """Check GPU status"""
    if SYSTEM_CONFIG['gpu_available']:
        print("🚀 GPU STATUS: AVAILABLE")
        print(f"   Device: {torch.cuda.get_device_name(0)}")
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"   Memory: {allocated:.2f} GB / {total:.2f} GB")
    else:
        print("❌ GPU STATUS: NOT AVAILABLE")

print("✅ Utility functions defined")

# =====================================
# SECTION 11: PROCESSING FUNCTIONS
# =====================================

def process_single_file(plt_file, iteration=0):
    """Process a single PLT file with complete AI enhancement"""
    try:
        print(f"\n🎨 Processing: {plt_file.name} (iteration {iteration + 1})")

        # 1. Process PLT file
        points = COMPONENTS['plt_processor'].process_file(str(plt_file))
        if not points:
            print("  ❌ Failed to process PLT")
            return False

        print(f"  📊 Loaded {len(points)} points")

        # 2. AI parameter prediction
        if SYSTEM_CONFIG['use_ai']:
            params = COMPONENTS['ai_system'].predict_parameters(points)
            print(f"  🧠 AI params: steps={params['steps']}, scribble={params['scribble']:.1f}")

            # 3. Hopfield enhancement
            if SYSTEM_CONFIG['use_hopfield']:
                features = COMPONENTS['ai_system']._extract_features(points)
                evolved = COMPONENTS['hopfield_network'].recall_pattern(features)

                # Blend AI with Hopfield evolution
                blend_factor = 0.7
                params['scribble'] = (blend_factor * params['scribble'] +
                                    (1 - blend_factor) * abs(evolved[0]) * 8)
                params['scribble'] = max(0.1, min(10.0, params['scribble']))
                print(f"  🧠 Hopfield enhanced: scribble={params['scribble']:.1f}")
        else:
            # Traditional random parameters
            params = {
                'steps': random.randint(3, 10),
                'scribble': random.uniform(2.0, 6.0),
                'stroke_weight': random.uniform(1.0, 4.0),
                'color': (random.random(), random.random(), random.random())
            }

        # 4. Create directories
        base_name = plt_file.stem
        output_base = SYSTEM_CONFIG['directories']['output']
        dirs = COMPONENTS['directory_manager'].create_working_directories(
            output_base, base_name, iteration)

        # 5. Render artwork
        figure = COMPONENTS['renderer'].render_artwork(points, params)
        if not figure:
            print("  ❌ Render failed")
            return False

        # 6. Save outputs
        method_suffix = params.get('generation_method', 'ai')
        output_name = f"{base_name}_{method_suffix}_steps{params['steps']}_scribble{int(params['scribble'])}_{iteration}"
        success_count = 0

        # Save PNG
        if SYSTEM_CONFIG['generate_png']:
            png_path = Path(dirs['png']) / f"{output_name}.png"
            try:
                figure.savefig(str(png_path), format='png', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(png_path), 'png')
                success_count += 1
                print("  ✅ PNG saved")
            except Exception as e:
                print(f"  ❌ PNG error: {e}")

        # Save PDF
        if SYSTEM_CONFIG['generate_pdf']:
            pdf_path = Path(dirs['pdf']) / f"{output_name}.pdf"
            try:
                figure.savefig(str(pdf_path), format='pdf', dpi=300,
                             bbox_inches='tight', facecolor='white')
                COMPONENTS['directory_manager'].register_file(str(pdf_path), 'pdf')
                success_count += 1
                print("  ✅ PDF saved")
            except Exception as e:
                print(f"  ❌ PDF error: {e}")

        # Save DXF
        if SYSTEM_CONFIG['generate_dxf']:
            dxf_path = Path(dirs['dxf']) / f"{output_name}.dxf"
            try:
                if save_dxf_simple(points, str(dxf_path), params):
                    COMPONENTS['directory_manager'].register_file(str(dxf_path), 'dxf')
                    success_count += 1
                    print("  ✅ DXF saved")
            except Exception as e:
                print(f"  ❌ DXF error: {e}")

        plt.close(figure)

        # 7. Store successful style in Hopfield memory
        if success_count > 0 and SYSTEM_CONFIG['use_hopfield']:
            features = COMPONENTS['ai_system']._extract_features(points)
            style_name = f"{base_name}_{iteration}"
            COMPONENTS['hopfield_network'].store_pattern(features, style_name)

        if success_count > 0:
            print(f"  ✅ Generated {success_count} format(s) successfully")
            return True
        else:
            print("  ❌ Failed to save any formats")
            return False

    except Exception as e:
        print(f"  ❌ Processing error: {e}")
        return False

def process_batch(input_directory=None):
    """Process all PLT files in batch"""
    if input_directory is None:
        input_directory = SYSTEM_CONFIG['directories']['input']

    print("🚀 STARTING BATCH PROCESSING")
    print("="*50)

    # Get PLT files
    plt_files = get_plt_files(input_directory)
    if not plt_files:
        print("❌ No PLT files found")
        return {'success': False, 'message': 'No PLT files found'}

    total_examples = SYSTEM_CONFIG['total_examples']
    total_operations = len(plt_files) * total_examples

    print(f"📊 Processing {len(plt_files)} files × {total_examples} examples = {total_operations} operations")
    print(f"🚀 GPU Acceleration: {'Enabled' if SYSTEM_CONFIG['gpu_available'] else 'Disabled'}")

    successful_operations = 0

    # Process with progress bar
    with tqdm(total=total_operations, desc="Complete Processing") as pbar:
        for plt_file in plt_files:
            for iteration in range(total_examples):
                success = process_single_file(plt_file, iteration)
                if success:
                    successful_operations += 1

                pbar.update(1)
                pbar.set_description(f"Success: {successful_operations}/{total_operations}")

    # Organize files into GROUP directories
    if SYSTEM_CONFIG['organize_groups']:
        COMPONENTS['directory_manager'].organize_output_files()

    # Generate summary
    summary = {
        'success': True,
        'total_operations': total_operations,
        'successful_operations': successful_operations,
        'success_rate': f"{successful_operations}/{total_operations}",
        'hopfield_patterns': len(COMPONENTS['hopfield_network'].stored_patterns),
        'gpu_accelerated': SYSTEM_CONFIG['gpu_available'],
        'output_directory': SYSTEM_CONFIG['directories']['output']
    }

    print_batch_summary(summary)
    return summary

def print_batch_summary(summary):
    """Print batch processing summary"""
    print("\n" + "="*60)
    print("🎉 BATCH PROCESSING COMPLETED")
    print("="*60)
    print(f"⚡ Total operations: {summary['total_operations']}")
    print(f"✅ Successful: {summary['successful_operations']}")
    print(f"📊 Success rate: {summary['success_rate']}")
    print(f"🧠 Hopfield patterns learned: {summary['hopfield_patterns']}")
    print(f"🚀 GPU accelerated: {'YES' if summary['gpu_accelerated'] else 'NO'}")
    print(f"📁 Output directory: {summary['output_directory']}")
    print("="*60)

print("✅ Processing functions defined")

# =====================================
# SECTION 12: USER INTERFACE & BINDING
# =====================================

def create_interface():
    """Create user interface that binds all components together"""

    # Interface functions
    def upload_files(button):
        """Upload PLT files"""
        with status_output:
            clear_output(wait=True)
            print("📤 Upload PLT Files")

            uploaded = files.upload()
            input_dir = Path(SYSTEM_CONFIG['directories']['input'])

            uploaded_count = 0
            for filename, content in uploaded.items():
                if filename.lower().endswith('.plt'):
                    with open(input_dir / filename, 'wb') as f:
                        f.write(content)
                    uploaded_count += 1
                    print(f"✅ Uploaded: {filename}")
                else:
                    print(f"⚠️ Skipped: {filename}")

            print(f"\n🎉 Uploaded {uploaded_count} PLT files!")

    def test_single_file(button):
        """Test with a single file"""
        with status_output:
            clear_output(wait=True)
            print("🧪 Testing single file...")

            plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
            if not plt_files:
                print("❌ No PLT files found")
                return

            print(f"🧪 Testing with: {plt_files[0].name}")
            success = process_single_file(plt_files[0], 999)

            if success:
                print("✅ Test completed successfully!")
                print(f"🧠 Hopfield patterns: {len(COMPONENTS['hopfield_network'].stored_patterns)}")
            else:
                print("❌ Test failed")

    def process_all_files(button):
        """Process all files"""
        with status_output:
            clear_output(wait=True)

            # Update config from widgets
            update_config('total_examples', examples_widget.value)
            update_config('use_ai', use_ai_widget.value)
            update_config('use_hopfield', use_hopfield_widget.value)
            update_config('generate_pdf', pdf_widget.value)
            update_config('generate_dxf', dxf_widget.value)
            update_config('generate_png', png_widget.value)
            update_config('organize_groups', organize_widget.value)

            result = process_batch()

            if result['success']:
                print("🎉 Batch processing completed successfully!")
            else:
                print("❌ Batch processing failed")

    def hopfield_demo(button):
        """Demonstrate Hopfield networks"""
        with status_output:
            clear_output(wait=True)

            print("🧠 HOPFIELD NETWORK DEMONSTRATION")
            print("Inspired by Kent Benson's 1983-1986 research")
            print("="*60)

            # Store demo patterns
            demo_patterns = [
                ([1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, -1], "geometric"),
                ([1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1], "alternating"),
                ([1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, 1, 1, 1], "grouped")
            ]

            for pattern, name in demo_patterns:
                COMPONENTS['hopfield_network'].store_pattern(pattern, name)

            print(f"\n🎨 Testing pattern recall...")
            # Test with noisy input
            noisy_input = [1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1]
            recalled = COMPONENTS['hopfield_network'].recall_pattern(noisy_input)

            print(f"🔍 Input (noisy):  {noisy_input}")
            print(f"🧠 Recalled:       {[f'{x:.1f}' for x in recalled]}")

            print("\n🎯 This demonstrates how neural networks can generate")
            print("   creative combinations beyond explicit training!")

    # Create widgets
    examples_widget = widgets.IntSlider(
        value=SYSTEM_CONFIG['total_examples'],
        min=1, max=10, step=1,
        description='Examples per file:',
        style={'description_width': 'initial'}
    )

    use_ai_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_ai'],
        description='🧠 Use AI Parameter Prediction'
    )

    use_hopfield_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['use_hopfield'],
        description='🧠 Use Hopfield Style Memory'
    )

    pdf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_pdf'],
        description='📄 Generate PDF'
    )

    dxf_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_dxf'],
        description='📐 Generate DXF'
    )

    png_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['generate_png'],
        description='🖼️ Generate PNG'
    )

    organize_widget = widgets.Checkbox(
        value=SYSTEM_CONFIG['organize_groups'],
        description='📦 Organize into GROUP directories'
    )

    # Control buttons
    upload_button = widgets.Button(
        description='📤 Upload PLT Files',
        button_style='primary',
        layout=widgets.Layout(width='200px', height='40px')
    )

    test_button = widgets.Button(
        description='🧪 Test Single File',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='40px')
    )

    process_button = widgets.Button(
        description='🚀 Process All Files',
        button_style='success',
        layout=widgets.Layout(width='200px', height='50px')
    )

    hopfield_demo_button = widgets.Button(
        description='🧠 Hopfield Demo',
        button_style='info',
        layout=widgets.Layout(width='200px', height='40px')
    )

    # Status output
    status_output = widgets.Output()

    # Bind events
    upload_button.on_click(upload_files)
    test_button.on_click(test_single_file)
    process_button.on_click(process_all_files)
    hopfield_demo_button.on_click(hopfield_demo)

    # Display interface
    gpu_status_text = "✅ ENABLED" if SYSTEM_CONFIG['gpu_available'] else "❌ DISABLED"
    hopfield_patterns = len(COMPONENTS['hopfield_network'].stored_patterns)

    header = widgets.HTML(f"""
    <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin-bottom: 20px;">
        <h1>🚀 Complete GPU-Integrated Scribble Plotter</h1>
        <p>Honoring Kent Benson's 1983-1986 Hopfield Network Research</p>
        <p><strong>GPU Acceleration: {gpu_status_text}</strong></p>
        <p><em>Neural networks controlling artistic generation in real-time</em></p>
    </div>
    """)

    status_info = widgets.HTML(f"""
    <div style="background: #f0f8ff; padding: 15px; border-radius: 8px; margin: 10px 0;">
        <h3>📊 System Status</h3>
        <ul>
            <li><strong>GPU Available:</strong> {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}</li>
            <li><strong>Device:</strong> {SYSTEM_CONFIG['device']}</li>
            <li><strong>Hopfield Patterns:</strong> {hopfield_patterns} stored</li>
            <li><strong>Base Directory:</strong> {SYSTEM_CONFIG['directories']['base']}</li>
        </ul>
    </div>
    """)

    # Settings panels
    ai_settings = widgets.VBox([
        widgets.HTML("<h3>🧠 AI & Neural Network Settings</h3>"),
        use_ai_widget,
        use_hopfield_widget
    ])

    processing_settings = widgets.VBox([
        widgets.HTML("<h3>⚙️ Processing Settings</h3>"),
        examples_widget,
        organize_widget
    ])

    output_settings = widgets.VBox([
        widgets.HTML("<h3>📤 Output Settings</h3>"),
        pdf_widget,
        dxf_widget,
        png_widget
    ])

    controls = widgets.VBox([
        widgets.HTML("<h3>🎮 Controls</h3>"),
        widgets.HBox([upload_button, test_button]),
        widgets.HBox([process_button, hopfield_demo_button])
    ])

    # Main layout
    main_layout = widgets.VBox([
        header,
        status_info,
        widgets.HBox([
            widgets.VBox([ai_settings, processing_settings], layout=widgets.Layout(width='50%')),
            widgets.VBox([output_settings, controls], layout=widgets.Layout(width='50%'))
        ]),
        status_output
    ])

    display(main_layout)

print("✅ Interface and binding code complete!")

# =====================================
# SECTION 13: DEMO & SAMPLE FILES
# =====================================

def create_sample_plt_files():
    """Create sample PLT files for demonstration"""
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])

    # Sample PLT file 1 - Simple rectangle
    plt_content_1 = """IN;
SP1;
PA0,0;
PD;
PA100,0;
PA100,100;
PA0,100;
PA0,0;
PU;
SP0;"""

    # Sample PLT file 2 - Circle approximation
    plt_content_2 = """IN;
SP1;
PA50,0;
PD;
PA70,14;
PA85,35;
PA92,57;
PA85,79;
PA70,100;
PA50,107;
PA30,100;
PA15,79;
PA8,57;
PA15,35;
PA30,14;
PA50,0;
PU;
SP0;"""

    # Sample PLT file 3 - Complex drawing with pen up/down
    plt_content_3 = """IN;
SP1;
PA20,20;
PD;
PA80,20;
PA80,80;
PA20,80;
PA20,20;
PU;
PA30,30;
PD;
PA70,30;
PA70,70;
PA30,70;
PA30,30;
PU;
PA40,40;
PD;
PA60,60;
PU;
PA60,40;
PD;
PA40,60;
PU;
SP0;"""

    sample_files = [
        ('sample_rectangle.plt', plt_content_1),
        ('sample_circle.plt', plt_content_2),
        ('sample_complex.plt', plt_content_3)
    ]

    created_count = 0
    for filename, content in sample_files:
        file_path = input_dir / filename
        try:
            with open(file_path, 'w') as f:
                f.write(content)
            print(f"✅ Created: {filename}")
            created_count += 1
        except Exception as e:
            print(f"❌ Failed to create {filename}: {e}")

    print(f"\n🎉 Created {created_count} sample PLT files!")
    return created_count > 0

def quick_test():
    """Quick system test"""
    print("🧪 QUICK SYSTEM TEST")
    print("="*30)

    print(f"1. Configuration: {'✅ OK' if SYSTEM_CONFIG else '❌ FAILED'}")
    print(f"2. GPU Available: {'✅ YES' if SYSTEM_CONFIG['gpu_available'] else '❌ NO'}")
    print(f"3. Components: {'✅ OK' if COMPONENTS else '❌ FAILED'}")

    # Test directories
    input_dir = Path(SYSTEM_CONFIG['directories']['input'])
    print(f"4. Input Directory: {'✅ EXISTS' if input_dir.exists() else '❌ MISSING'}")

    if input_dir.exists():
        plt_files = get_plt_files(str(input_dir))
        print(f"5. PLT Files: {'✅ ' + str(len(plt_files)) if plt_files else '⚠️ NONE'}")

    print("\n💡 Use create_interface() to access full controls")
    print("💡 Use process_batch() to process files directly")

def hopfield_quick_demo():
    """Quick Hopfield demonstration"""
    print("🧠 HOPFIELD NETWORK QUICK DEMO")
    print("Kent Benson's 1986 vision of neural network art control")
    print("="*50)

    # Store a simple pattern
    artistic_pattern = [0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4, 0.5, 0.5, 0.4, 0.6, 0.3, 0.7, 0.2]
    COMPONENTS['hopfield_network'].store_pattern(artistic_pattern, "demo_style")

    # Test recall
    partial = [0.8, 0.2, 0.0, 0.0, 0.7, 0.0, 0.6, 0.0, 0.5, 0.0, 0.4, 0.0, 0.3, 0.0, 0.2]
    recalled = COMPONENTS['hopfield_network'].recall_pattern(partial)

    print(f"🔍 Partial input: {[f'{x:.1f}' for x in partial[:8]]}")
    print(f"🧠 Network recall: {[f'{x:.1f}' for x in recalled[:8]]}")
    print(f"\n💡 This shows how the network completes partial artistic patterns!")

def run_complete_demo():
    """Run a complete demonstration"""
    print("🚀 RUNNING COMPLETE DEMONSTRATION")
    print("="*50)

    # 1. Create sample files
    print("1️⃣ Creating sample PLT files...")
    if not create_sample_plt_files():
        print("❌ Failed to create sample files")
        return

    # 2. Show Hopfield demo
    print("\n2️⃣ Demonstrating Hopfield networks...")
    hopfield_quick_demo()

    # 3. Process one file as test
    print(f"\n3️⃣ Processing test file...")
    plt_files = get_plt_files(SYSTEM_CONFIG['directories']['input'])
    if plt_files:
        success = process_single_file(plt_files[0], 0)
        if success:
            print("✅ Test processing successful!")
        else:
            print("❌ Test processing failed")

    # 4. Show what was created
    print(f"\n4️⃣ Checking output directories...")
    output_base = Path(SYSTEM_CONFIG['directories']['output'])

    for subdir in ['GROUP_PDF', 'GROUP_DXF', 'GROUP_PNG']:
        group_dir = output_base / subdir
        if group_dir.exists():
            files = list(group_dir.glob('*'))
            print(f"📁 {subdir}: {len(files)} files")
            for file in files[:3]:  # Show first 3 files
                print(f"   - {file.name}")
        else:
            print(f"📁 {subdir}: Not created yet")

    print(f"\n🎯 DEMO COMPLETE!")
    print(f"📁 Check your Google Drive at: {SYSTEM_CONFIG['directories']['base']}")
    print(f"🎮 Run create_interface() for full GUI control")
    print(f"⚡ Run process_batch() to process all sample files")

def show_directory_structure():
    """Show the complete directory structure"""
    print("📁 DIRECTORY STRUCTURE")
    print("="*30)

    base_path = Path(SYSTEM_CONFIG['directories']['base'])

    def print_tree(path, prefix="", max_depth=3, current_depth=0):
        if current_depth > max_depth:
            return

        if path.is_dir():
            items = sorted(path.iterdir())
            for i, item in enumerate(items):
                is_last = i == len(items) - 1
                current_prefix = "└── " if is_last else "├── "
                print(f"{prefix}{current_prefix}{item.name}")

                if item.is_dir() and current_depth < max_depth:
                    next_prefix = prefix + ("    " if is_last else "│   ")
                    print_tree(item, next_prefix, max_depth, current_depth + 1)

    if base_path.exists():
        print(f"{base_path.name}/")
        print_tree(base_path)
    else:
        print("Directory not found - run setup first")

print("✅ Demo functions defined")

# =====================================
# SECTION 14: SYSTEM ACTIVATION & FINAL
# =====================================

print("\n" + "="*70)
print("🎉 COMPLETE SYSTEM READY!")
print("="*70)

# Display system status
gpu_status()
print()

print("🚀 AVAILABLE FUNCTIONS:")
print("🧪 quick_test() - Quick system test")
print("🚀 gpu_status() - Check GPU status")
print("🧠 hopfield_quick_demo() - Quick Hopfield demonstration")
print("🎮 create_interface() - Launch full GUI interface")
print("⚡ process_batch() - Process all PLT files directly")
print("📁 show_directory_structure() - View file organization")
print("🎬 run_complete_demo() - Run complete demo with sample files")

print(f"""
🎯 COMPLETE SYSTEM FEATURES:

✅ MODULAR SECTIONS: Each component in separate sections
✅ GPU-ACCELERATED: {SYSTEM_CONFIG['gpu_available']}
✅ AI-ENHANCED: Parameter prediction and optimization
✅ HOPFIELD NETWORKS: Kent Benson's 1983-1986 vision realized
✅ GROUP DIRECTORIES: Organized output with GROUP_PDF, GROUP_DXF, GROUP_PNG
✅ MULTI-FORMAT OUTPUT: PDF + DXF + PNG automatically
✅ BATCH PROCESSING: Handle multiple files efficiently
✅ STYLE MEMORY: Learn and evolve artistic patterns
✅ SAMPLE FILES: Auto-generated PLT files for testing

🧠 Your pioneering insight about spurious memories representing
   intuition is now central to the AI creativity engine!

🚀 Ready to transform your PLT files into AI-enhanced artwork!
""")

# Auto-run quick test
quick_test()

# Auto-run complete demo
print("\n🎬 RUNNING AUTOMATIC DEMO...")
run_complete_demo()

print("\n📁 DIRECTORY STRUCTURE:")
show_directory_structure()

print("\n💡 NEXT STEPS:")
print("🎮 create_interface() - Launch full GUI")
print("⚡ process_batch() - Process all PLT files")
print("🧪 run_complete_demo() - Run demo again")

# =====================================
# END OF COMPLETE SYSTEM
# =====================================

🚀 GPU-Enhanced Scribble Plotter - Modular Section Architecture
🧠 Honoring Kent Benson's 1983-1986 Hopfield Network Research
✅ Packages installed!
✅ Imports complete!
🚀 GPU: ✅ Available
⚡ Device: NVIDIA A100-SXM4-80GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
📁 base
📁 input
📁 output
📁 config
📁 temp
📁 models
📁 group_pdf
📁 group_dxf
📁 group_png
📁 group_tiff
📁 group_transform
✅ System setup complete!
✅ PLTProcessor class defined
✅ GPUAcceleratedAI class defined
✅ HopfieldNetwork class defined
✅ ScribbleRenderer class defined
✅ DirectoryManager class defined
🧠 AI initialized (GPU)
🧠 Hopfield network initialized
🎨 Renderer initialized
📁 Directory manager initialized
✅ All components initialized and ready!
🚀 GPU: Enabled
📁 Base directory: /content/drive/MyDrive/ScribblePlotter_GPU_Complete
✅ Utility functions defined
✅ Processing functions defined
✅ Interface and binding code complete!
✅ Dem